# 4D Gaussian Splatting with native 4D primitives (fudan-zvg/4d-gaussian-splatting): D-NeRF monocular benchmark

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riccardolanza05/dynamic-gaussian-splatting-benchmark/blob/main/notebooks/03_4dgs_native4d_fudan_dnerf.ipynb)

Part of **dynamic-gaussian-splatting-benchmark**, a controlled comparison of three dynamic Gaussian Splatting methods on the eight scenes of the monocular D-NeRF synthetic dataset, run on a free-tier Google Colab Tesla T4. The method is trained with its official training code; a benchmark monitor attached to its training loop records PSNR, SSIM, LPIPS, evaluation L1, training time, iteration count, number of Gaussians, peak VRAM and model storage into one JSON file per run.

**AI disclosure.** This notebook, the rest of the repository and part of the code used to extract the metrics (the benchmark monitor, the per-scene preparation and the analysis scripts) were generated with AI assistance (Anthropic's Claude), starting from the code of the official repository of each method. The training code of the method itself is the official one, cloned at run time; the only changes are the monitor hook appended to `train.py`, configuration overrides and environment fixes, all made by the cells below.

**Method.** Real-time Photorealistic Dynamic Scene Representation and Rendering with 4D Gaussian Splatting, Yang et al., ICLR 2024 ([paper](https://arxiv.org/abs/2310.10642), [code](https://github.com/fudan-zvg/4d-gaussian-splatting)), extended in *4D Gaussian Splatting: Modeling Dynamic Scenes with Native 4D Primitives* ([paper](https://arxiv.org/abs/2412.20720)). No canonical space: every primitive is a 4D Gaussian with a finite temporal extent, sliced at the rendering time.

Two protocols are supported:

* **Protocol A, equal iterations**: every method trains for 30 000 optimisation steps, metrics sampled every 1 000 steps.
* **Protocol B, equal quality**: training stops when the evaluation L1 reaches a per-scene target shared by the three notebooks, metrics sampled every 30 seconds.

Results, methodology and calibration are documented in the repository `README.md` and in `docs/`.

The repository ships neither a viewer nor a `render.py`. Part 2 therefore writes a community `render.py` for the final metrics and renders videos offline, following the approach of [PR #60](https://github.com/fudan-zvg/4d-gaussian-splatting/pull/60) but reading the time range from the model instead of the hard-coded `[0, 10]` range, which is wrong for D-NeRF (`[0, 1]`).

**Notebook structure**

*Part 1: configuration, setup and training*

0. User configuration and path resolution
1. Environment and repository setup
2. CUDA module and environment fixes
3. Dataset download and per-scene preparation (derived YAML configs)
4. Training with incremental benchmarking (single scene or loop, Protocol B calibration, summary)

*Part 2: evaluation and visualisation*

5. Rendering and final metrics
6. Dense temporal rendering and video export
7. Real-time streaming viewer

---

# Part 1: configuration, setup and training

This part is safe to run unattended. It configures the run, prepares the environment and the dataset, installs the benchmark instrumentation and trains one scene or a list of scenes. It produces model checkpoints and one JSON summary per run.

**Do not run Part 2 during a training loop.** It renders, computes metrics and starts a streaming server, which compete for the GPU and would corrupt the training-time and peak-VRAM measurements.

---

## 0. User configuration

Everything that changes between runs is set in cell 0.1; no other cell needs editing.

**Storage.** `STORAGE_MODE = "drive"` writes the benchmark JSON (and, with `KEEP_MODEL_ON_DRIVE = True`, the model) to Google Drive, so a run survives a runtime restart and the loop can resume. `"local"` keeps everything in `/content`, which is lost when the session ends.

**What to run.** `RUN_MODE = "single"` trains `SCENE`; `"loop"` trains every scene in `SCENES_TO_RUN`. Completed runs are skipped, so re-running a loop after a disconnection resumes from the first missing scene.

**Which protocol.**

| | Protocol A, `"iterations"` | Protocol B, `"target_eval_loss"` |
|---|---|---|
| Stopping criterion | fixed budget `MAX_ITERATIONS` | per-scene target in `TARGET_EVAL_LOSS_PER_SCENE` |
| Metric sampling | every `EVAL_EVERY_N_ITERS` iterations | every `EVAL_EVERY_N_MINUTES` minutes |
| Question answered | at equal budget, which method reconstructs best? | to reach the same quality, what does each method cost? |

**Workflow.** The Protocol B target must be reachable by every method, so it is calibrated from Protocol A: run the Protocol A loop on all scenes in all three notebooks, read the candidates printed by cell 4.3, take the per-scene maximum across the three methods and use the same dictionary in all three notebooks. The calibrated targets used in this study are already filled in.

**Uniform conventions.** All methods are evaluated at 800×800 on the full 20-view test split, on a black background, with LPIPS computed by the VGG backbone. Black is imposed by fudan-zvg: `scene/cameras.py` premultiplies the ground truth by the alpha channel, so its ground truth is black whatever the flag says, and rendering on white against it makes training collapse.

In [ ]:
# 0.1  User configuration: edit this cell, then run Part 1 top to bottom.

# Output storage: "drive" is persistent (survives restarts), "local" lives in /content.
STORAGE_MODE = "drive"

DRIVE_OUTPUT_ROOT = "/content/drive/MyDrive/4DGS_fudan_output"
LOCAL_OUTPUT_ROOT = "/content/4DGS_fudan_output"

# "single" trains SCENE only; "loop" trains every scene in SCENES_TO_RUN, in order.
RUN_MODE = "loop"

SCENE = "bouncingballs"

# Used only when RUN_MODE == "loop"; trim the list for a partial loop.
SCENES_TO_RUN = ["bouncingballs", "hellwarrior", "hook", "jumpingjacks",
                 "mutant", "standup", "trex"]
SCENES_NOT_TO_RUN = ["lego"]   # informational only: lego is left out of SCENES_TO_RUN because fudan-zvg cannot train it

# Skip runs that are already complete (this makes the loop resumable).
SKIP_TRAINING_IF_TRAINED = True

# False: the model stays in /content and only the benchmark JSON reaches Drive
#        (model storage is measured before the local copy is discarded).
# True:  the model is also copied to Drive, so Part 2 can run later without retraining.
KEEP_MODEL_ON_DRIVE = False
LOCAL_TRAIN_ROOT = "/content/train_outputs"

# Benchmarking protocol: "iterations" (Protocol A) or "target_eval_loss" (Protocol B).
TRAINING_MODE = "iterations"

# Protocol A: fixed iteration budget
MAX_ITERATIONS     = 30000   # optimisation budget
EVAL_EVERY_N_ITERS = 1000    # sample the metrics every N iterations

# Protocol B: fixed quality target
# "eval_loss" stops on the per-scene L1 target below, "psnr" on TARGET_PSNR.
TARGET_METRIC = "eval_loss"
TARGET_PSNR   = 30.0    # used only when TARGET_METRIC == "psnr"

# Per-scene L1 target, identical in the three notebooks and calibrated from the Protocol A
# results of all three methods (cell 4.3, docs/PROTOCOL_B_CALIBRATION.md).
# A scene left at None is skipped by the Protocol B loop.
TARGET_EVAL_LOSS_PER_SCENE = {
    # target = 1.05 x max over methods of the minimum eval L1 on the Protocol A test curve.
    # lego: fudan-zvg is excluded (its training aborts on that scene), so Wu et al. set the target.
    "bouncingballs": 0.005083,   # fudan  0.004841 @18000  x1.05
    "hellwarrior":   0.005358,   # fudan  0.005103 @10000  x1.05
    "hook":          0.006253,   # fudan  0.005956 @4000   x1.05
    "jumpingjacks":  0.004610,   # fudan  0.004391 @4000   x1.05  (partial run, 15000 it)
    "lego":          0.013466,   # Wu     0.012825 @10000  x1.05  (fudan excluded)
    "mutant":        0.003106,   # fudan  0.002958 @11000  x1.05
    "standup":       0.002168,   # fudan  0.002065 @10000  x1.05
    "trex":          0.005793,   # fudan  0.005517 @4000   x1.05  (partial run, 6000 it)
}

EVAL_EVERY_N_MINUTES  = 0.5     # sample the metrics every T minutes
SAFETY_MAX_ITERATIONS = 60000   # hard cap, so a run can never last forever
# The target is ignored below this iteration, so the baseline sample cannot stop the run.
MIN_ITERATIONS_BEFORE_STOP = 1000
# The target must hold for this many consecutive samples (test metrics fluctuate).
TARGET_CONSECUTIVE_HITS = 2

# Evaluation settings (shared by both protocols)
# "l1": mean L1 on the test split; "l1_dssim": (1-w)*L1 + w*(1-SSIM). Both are always recorded;
# this selects the one mirrored by "eval_loss" and used by the Protocol B stop.
EVAL_LOSS_KIND = "l1"
LAMBDA_DSSIM   = 0.2      # weight w of the "l1_dssim" variant

LPIPS_NET      = "vgg"    # LPIPS backbone, as reported in all three papers
MAX_EVAL_VIEWS = 0        # 0 = every test view; must stay 0 for the shared L1 target
EVAL_AT_FIRST_ITERATION = True   # record a baseline point at the first iteration

# Extra flags appended verbatim to the training command (usually empty)
EXTRA_TRAIN_ARGS = ""

# Views consumed per iteration; overwritten from the scene YAML (1 to 24 in the official configs)
BATCH_SIZE = 2

METHOD_NAME = "4D Gaussian Splatting, native 4D primitives (fudan-zvg)"
REPO_DIR = "/content/4d-gaussian-splatting"
D_NERF_SCENES = ["bouncingballs", "hellwarrior", "hook", "jumpingjacks",
                 "lego", "mutant", "standup", "trex"]

assert STORAGE_MODE in ("drive", "local")
assert RUN_MODE in ("single", "loop")
assert TRAINING_MODE in ("iterations", "target_eval_loss")
assert EVAL_LOSS_KIND in ("l1", "l1_dssim")
assert TARGET_METRIC in ("psnr", "eval_loss")
assert SCENE in D_NERF_SCENES, "Unknown scene: %s" % SCENE
assert all(s in D_NERF_SCENES for s in SCENES_TO_RUN)
print("Configuration accepted.")
print("Method   : %s" % METHOD_NAME)
print("Run mode : %s" % RUN_MODE)
print("Protocol : %s" % TRAINING_MODE)
print("Scenes   : %s" % (SCENE if RUN_MODE == "single" else ", ".join(SCENES_TO_RUN)))

In [ ]:
# 0.2  Path resolution: every path is a function of (scene, protocol); Drive is mounted only if requested.
import glob
import json
import os

if STORAGE_MODE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = DRIVE_OUTPUT_ROOT
else:
    OUTPUT_ROOT = LOCAL_OUTPUT_ROOT

os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(LOCAL_TRAIN_ROOT, exist_ok=True)
TRAINED_GLOB = ("chkpnt*.pth",)


def run_name(scene, mode):
    """Folder name for one (scene, protocol) pair. Runs never collide."""
    if mode == "iterations":
        return "%s_iters%d" % (scene, MAX_ITERATIONS)
    target = TARGET_EVAL_LOSS_PER_SCENE.get(scene)
    tag = ("%g" % target).replace(".", "p") if target is not None else "NA"
    return "%s_loss%s" % (scene, tag)


def paths_for(scene, mode):
    """All output paths for one run.

    The benchmark JSON always lives on OUTPUT_ROOT (Drive when selected), while
    the model goes to Drive only if KEEP_MODEL_ON_DRIVE, otherwise to local
    disk. Metrics are what the benchmark needs; the checkpoints are optional.
    """
    name = run_name(scene, mode)
    model_root = OUTPUT_ROOT if KEEP_MODEL_ON_DRIVE else LOCAL_TRAIN_ROOT
    model_path = os.path.join(model_root, name)
    bench_dir = os.path.join(OUTPUT_ROOT, name, "benchmark")
    return {
        "scene": scene,
        "mode": mode,
        "run_name": name,
        "model_path": model_path,
        "drive_model_path": os.path.join(OUTPUT_ROOT, name),
        "benchmark_dir": bench_dir,
        "bench_json": os.path.join(bench_dir, "benchmark_%s.json" % mode),
        "bench_config": os.path.join(bench_dir, "benchmark_config.json"),
        "source_path": os.path.join(REPO_DIR, "dataset_dir", "data", scene),
    }


def is_model_trained(model_path):
    """True when the model files of that run are present on disk."""
    return len(glob.glob(os.path.join(model_path, *TRAINED_GLOB))) > 0


TERMINAL_STATUS = ("finished", "stopped_on_target", "stopped_on_safety_cap")


def run_is_complete(p):
    """True when this run has already produced a finished benchmark JSON.

    Completion is judged on the JSON rather than on the model, because with
    KEEP_MODEL_ON_DRIVE = False the model does not survive the runtime. This is
    what makes the loop resumable after a disconnection or a GPU-quota stop.
    """
    if not os.path.exists(p["bench_json"]):
        return False
    try:
        with open(p["bench_json"], "r") as handle:
            summary = json.load(handle)
    except Exception:
        return False
    if summary.get("status") not in TERMINAL_STATUS:
        return False
    if KEEP_MODEL_ON_DRIVE and not is_model_trained(p["drive_model_path"]):
        return False
    return True


def activate_scene(scene, mode=None):
    """Point the module-level variables at one scene. Part 2 reads these."""
    global SCENE, SOURCE_PATH, MODEL_OUTPUT_PATH, MODEL_DIR
    global BENCHMARK_DIR, BENCH_JSON_PATH, BENCH_CONFIG_PATH, RUN_NAME
    mode = mode or TRAINING_MODE
    p = paths_for(scene, mode)
    SCENE = scene
    SOURCE_PATH = p["source_path"]
    MODEL_OUTPUT_PATH = p["model_path"]
    MODEL_DIR = p["model_path"]          # alias used by Part 2
    BENCHMARK_DIR = p["benchmark_dir"]
    BENCH_JSON_PATH = p["bench_json"]
    BENCH_CONFIG_PATH = p["bench_config"]
    RUN_NAME = run_name(scene, mode)
    os.makedirs(BENCHMARK_DIR, exist_ok=True)
    return p


activate_scene(SCENE)

print("Metrics root : %s" % OUTPUT_ROOT)
print("Model folder : %s%s" % (MODEL_OUTPUT_PATH,
                               "" if KEEP_MODEL_ON_DRIVE else "   (not kept)"))
print("Benchmark    : %s" % BENCH_JSON_PATH)
print("Already done : %s" % run_is_complete(paths_for(SCENE, TRAINING_MODE)))

## 1. Environment and repository setup

In [ ]:
# Check the GPU assigned by Colab (e.g. Tesla T4)
!nvidia-smi

In [ ]:
%cd /content
!git clone https://github.com/fudan-zvg/4d-gaussian-splatting
%cd 4d-gaussian-splatting
!git submodule update --init --recursive

# Reuse Colab's PyTorch/CUDA and install the remaining dependencies with pip
!pip install -q plyfile tqdm scipy configargparse tensorboard pytorch-msssim torchmetrics ninja kornia

# The CUDA modules live in the repository root (not under submodules/)
!pip install -e diff-gaussian-rasterization
!pip install -e simple-knn
!pip install -e pointops2

## 2. CUDA module and environment fixes

* `simple-knn` needs `<float.h>` for its floating-point limits.
* The CUDA modules are reinstalled without `-e` (editable install); otherwise Python cannot find the compiled package with this repository's folder layout.
* Environment fixes for the current Colab image: `mmcv==1.6.0` and `open3d` have no wheel for its Python version. `mmcv` is not needed, and the unused `open3d` import in `scene/gaussian_model.py` is commented out. `lpips` and `plyfile` are installed explicitly because they are missing from `requirements.txt`.

In [ ]:
import os

os.chdir('/content/4d-gaussian-splatting/simple-knn')

filepath = 'simple_knn.cu'
if os.path.exists(filepath):
    with open(filepath, 'r') as f:
        content = f.read()
    if '#include <float.h>' not in content:
        with open(filepath, 'w') as f:
            f.write('#include <float.h>\n' + content)
        print('Patch applied successfully.')
    else:
        print('Patch already present.')
else:
    print(f"File {filepath} not found — make sure the module was cloned correctly.")

os.chdir('/content/4d-gaussian-splatting')

In [ ]:
import os

# Non-editable reinstall: avoids import errors caused by this repository's folder layout
os.chdir('/content/4d-gaussian-splatting/simple-knn')
os.makedirs('simple_knn', exist_ok=True)
!pip install -q .

os.chdir('/content/4d-gaussian-splatting/diff-gaussian-rasterization')
os.makedirs('diff_gaussian_rasterization', exist_ok=True)
!pip install -q .

os.chdir('/content/4d-gaussian-splatting')
print("✅ CUDA modules installed successfully!")

In [ ]:
# Environment fixes for the current Colab image:
# 1. requirements.txt pins mmcv==1.6.0, which has no wheel; it is not needed and is skipped.
# 2. The unused open3d import in scene/gaussian_model.py is commented out (no wheel either).
# 3. lpips and plyfile are used by the repository but missing from requirements.txt.
import os
import subprocess

get_ipython().system('pip install -q matplotlib lpips plyfile "imageio[ffmpeg]"')

target = os.path.join(REPO_DIR, "scene", "gaussian_model.py")
with open(target) as handle:
    lines = handle.readlines()

patched = False
for i, line in enumerate(lines):
    if line.strip() == "import open3d as o3d":
        lines[i] = ("# " + line +
                    "# ^ disabled: unused import, no wheel for this Python version\n")
        patched = True

if patched:
    with open(target, "w") as handle:
        handle.writelines(lines)
    print("open3d import neutralised in %s" % target)
else:
    print("open3d import already neutralised (or the line has changed).")

print()
for mod in ("plyfile", "lpips", "matplotlib", "imageio",
            "diff_gaussian_rasterization", "simple_knn", "torch"):
    r = subprocess.run("python -c 'import %s'" % mod, shell=True,
                       capture_output=True, text=True)
    print("  %-30s %s" % (mod, "OK" if r.returncode == 0 else "MISSING"))

r = subprocess.run(
    "cd %s && python -c 'from scene.gaussian_model import GaussianModel; print(\"OK\")'"
    % REPO_DIR, shell=True, capture_output=True, text=True)
print("\n  repo importable by train.py     %s"
      % (r.stdout.strip() or r.stderr.strip()[-300:]))

print("\n  torch version                   %s"
      % subprocess.run("python -c 'import torch; print(torch.__version__)'",
                       shell=True, capture_output=True, text=True).stdout.strip())

## 3. Dataset download

The D-NeRF synthetic dataset (all eight scenes) is downloaded once per runtime; the scene is selected per run.

In [ ]:
# Download the D-NeRF dataset once per runtime (the archive contains all eight scenes).
import os

os.chdir(REPO_DIR)

if not os.path.isdir("dataset_dir/data"):
    !mkdir -p dataset_dir
    !wget -q "https://www.dropbox.com/scl/fi/cdcmkufncwcikk1dzbgb4/data.zip?rlkey=n5m21i84v2b2xk6h7qgiu8nkg&dl=1" -O data.zip
    !unzip -q data.zip -d dataset_dir/
else:
    print("Dataset already present in this runtime, download skipped.")

DATASET_ROOT = os.path.join(REPO_DIR, "dataset_dir", "data")
available = sorted(d for d in os.listdir(DATASET_ROOT)
                   if os.path.isdir(os.path.join(DATASET_ROOT, d)))
print("Scenes available: %s" % available)

missing = [s for s in D_NERF_SCENES if s not in available]
assert not missing, "Scenes missing from the archive: %s" % missing

activate_scene(SCENE)
print("Active scene: %s -> %s" % (SCENE, SOURCE_PATH))

## 3.5 Per-scene preparation

This repository is configured through YAML, and `train.py` merges the file *over* the parsed arguments: values in the file override the command line, `--iterations` included. The budget is therefore written into a derived `<scene>_run.yaml`, leaving the official configs untouched. Two upstream defaults are changed there:

* `resolution: 2` halves the images (400×400 instead of 800×800) and is forced to 1.
* `exhaust_test: True` evaluates the whole test split every 500 iterations, adding a large and irregular cost to the measured training time. It is disabled; the benchmark monitor keeps `chkpnt_best.pth` with the same best-PSNR rule on its own schedule.

The per-scene `batch_size` of the official configs (1 to 24) is kept and recorded as `images_seen`. The background stays black: `scene/cameras.py` premultiplies the ground truth by its alpha channel, and a white background makes training collapse.

In [ ]:
# 3.5  Per-scene preparation (fudan-zvg). The YAML overrides the command line, so a derived
# <scene>_run.yaml sets the budget, resolution: 1 (800x800 instead of 400x400) and
# exhaust_test: False (the monitor keeps chkpnt_best.pth on its own schedule).
# The background must stay black: scene/cameras.py premultiplies the ground truth by alpha.
import glob
import os

from omegaconf import OmegaConf

EXHAUST_TEST = False


def prepare_scene(scene, mode):
    """Write the derived YAML for one scene; return extra CLI flags."""
    global RUN_CONFIG_PATH, BATCH_SIZE
    base = os.path.join(REPO_DIR, "configs", "dnerf", "%s.yaml" % scene)
    assert os.path.exists(base), "No official config for scene %s" % scene

    budget = MAX_ITERATIONS if mode == "iterations" else SAFETY_MAX_ITERATIONS
    p = paths_for(scene, mode)

    cfg = OmegaConf.load(base)
    cfg.ModelParams.source_path = p["source_path"]
    cfg.ModelParams.model_path = p["model_path"]
    cfg.ModelParams.resolution = 1
    cfg.OptimizationParams.iterations = int(budget)
    cfg.exhaust_test = bool(EXHAUST_TEST)

    RUN_CONFIG_PATH = os.path.join(REPO_DIR, "configs", "dnerf",
                                   "%s_run.yaml" % scene)
    OmegaConf.save(cfg, RUN_CONFIG_PATH)

    # Views per iteration, recorded as images_seen
    BATCH_SIZE = int(cfg.batch_size)
    return EXTRA_TRAIN_ARGS


def build_train_command(p, budget, extra):
    cmd = ('python train.py'
           + ' --config "%s"' % RUN_CONFIG_PATH
           + ' --eval'
           + ' --test_iterations 0'
           + ' --save_iterations %d' % budget)
    return cmd + ((' ' + extra) if extra else '')


def _folder_size_mb(root):
    total = 0
    for dirpath, _d, filenames in os.walk(root):
        for fn in filenames:
            try:
                total += os.path.getsize(os.path.join(dirpath, fn))
            except OSError:
                pass
    return total / 1048576.0


def _sum_patterns(model_path, patterns):
    files = []
    for pat in patterns:
        files += sorted(glob.glob(os.path.join(model_path, pat)))
    files = [f for f in files if os.path.isfile(f)]
    total = sum(os.path.getsize(f) for f in files)
    return files, total


# Only the model tensors of chkpnt_best.pth are counted, not the Adam state or the
# densification accumulators. capture() order for gaussian_dim == 4:
#   0 active_sh_degree   1 _xyz          2 _features_dc   3 _features_rest
#   4 _scaling           5 _rotation     6 _opacity       7 max_radii2D
#   8 xyz_gradient_accum 9 t_gradient_accum   10 denom    11 optimizer
#   12 spatial_lr_scale  13 _t          14 _scaling_t     15 _rotation_r
#   16 rot_4d            17 env_map     18 active_sh_degree_t
MODEL_ELEMENTS = (1, 2, 3, 4, 5, 6, 13, 14, 15)
REQUIRED_PATTERNS = ("cfg_args", "chkpnt_best.pth")


def _checkpoint_model_mb(ckpt_path):
    """Bytes of the model tensors only, optimizer state excluded."""
    import torch

    blob = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state = blob[0] if isinstance(blob, (tuple, list)) else blob
    total = 0
    for i in MODEL_ELEMENTS:
        el = state[i]
        if torch.is_tensor(el):
            total += el.numel() * el.element_size()
    return total / 1048576.0


def model_storage_report(model_path):
    files, total = _sum_patterns(model_path, REQUIRED_PATTERNS)
    report = {
        "required_files": [os.path.relpath(f, model_path) for f in files],
        "checkpoint_mb": round(total / 1048576.0, 4),
        "full_folder_mb": round(_folder_size_mb(model_path), 4),
    }
    ckpt = os.path.join(model_path, "chkpnt_best.pth")
    if os.path.exists(ckpt):
        report["model_storage_mb"] = round(_checkpoint_model_mb(ckpt), 4)
        report["optimizer_and_accum_mb"] = round(
            report["checkpoint_mb"] - report["model_storage_mb"], 4)
    else:
        report["model_storage_mb"] = report["checkpoint_mb"]
    return report


_ = prepare_scene(SCENE, TRAINING_MODE)
_cfg = OmegaConf.load(RUN_CONFIG_PATH)
print("Run config      : %s" % RUN_CONFIG_PATH)
print("resolution      : %s" % _cfg.ModelParams.resolution)
print("white_background: %s" % _cfg.ModelParams.white_background)
print("iterations      : %s" % _cfg.OptimizationParams.iterations)
print("exhaust_test    : %s" % _cfg.exhaust_test)
print("batch_size      : %s  (one iteration supervises %s views)"
      % (BATCH_SIZE, BATCH_SIZE))

## 4. Training with incremental benchmarking

**Instrumentation.** Each repository's training loop calls its own `training_report(...)` at every iteration. Instead of rewriting the loop, which would alter densification schedules, learning-rate ramps and stage handling, the notebook appends a few lines to `train.py` that wrap that function with a monitor. The upstream code path, hyper-parameters and command line stay untouched, a pristine copy is kept as `train.py.orig`, and without the `BENCH_CONFIG` environment variable `train.py` behaves exactly as upstream. The wrapper recovers the loop state through `inspect`, so it is robust to signature differences.

At every sampling point the monitor evaluates the current model on the full test split with the repository's own modules (`utils.image_utils.psnr`, `utils.loss_utils.ssim`, `utils.loss_utils.l1_loss`, bundled `lpipsPyTorch` with VGG) and records the metrics, the training time net of the benchmark overhead, the iteration count, the images seen, the number of Gaussians and the evaluation resolution. The JSON summary is rewritten atomically after each evaluation, so an interrupted run still leaves a valid file. Native periodic testing is disabled (`--test_iterations 0`) so that it does not add to the measured time.

**Protocol B cost.** A run stops only after the target has held for `TARGET_CONSECUTIVE_HITS` consecutive samples, so the termination time overstates the cost, by a different amount for each method. The figure to report is the **first crossing**, extracted by cell 4.4.

In [ ]:
# 4.0  Install the benchmark monitor and build the training driver
import json
import os
import shutil
import subprocess
import sys
import threading
import time

os.chdir(REPO_DIR)

MONITOR_SRC = r'''# benchmark_monitor.py -- generated by the notebook: incremental benchmark logger for __METHOD_NAME__.
# Wraps the repository training_report(...), which the training loop calls at every iteration,
# evaluates the current model on the test split with the repository metric modules and appends
# one record per sample to a JSON summary. In Protocol B it saves the model and stops on target.

import atexit
import inspect
import json
import os
import sys
import time

import torch

from utils.image_utils import psnr as _repo_psnr
from utils.loss_utils import ssim as _repo_ssim
from utils.loss_utils import l1_loss as _repo_l1


# Configuration written by the notebook, passed through $BENCH_CONFIG

_CFG_PATH = os.environ.get("BENCH_CONFIG", "")

with open(_CFG_PATH, "r") as _f:
    CFG = json.load(_f)

METHOD = CFG["method"]
SCENE = CFG["scene"]
MODE = CFG["mode"]                                   # "iterations" | "target_eval_loss"
JSON_PATH = CFG["json_path"]

EVAL_EVERY_N_ITERS = int(CFG.get("eval_every_n_iters", 2000))
EVAL_EVERY_N_SECONDS = float(CFG.get("eval_every_n_seconds", 600.0))
MAX_ITERATIONS = int(CFG.get("max_iterations", 30000))
SAFETY_MAX_ITERATIONS = int(CFG.get("safety_max_iterations", 100000))

# Protocol B stopping criterion: "eval_loss" (eval_loss <= target) or "psnr" (psnr >= target)
TARGET_METRIC = CFG.get("target_metric", "psnr")
TARGET_PSNR = float(CFG.get("target_psnr", 30.0))
TARGET_EVAL_LOSS = float(CFG.get("target_eval_loss", 0.0))
MIN_ITERATIONS_BEFORE_STOP = int(CFG.get("min_iterations_before_stop", 2000))
# Test metrics are not monotonic: the target must hold for several consecutive samples.
TARGET_CONSECUTIVE_HITS = int(CFG.get("target_consecutive_hits", 2))

# Evaluation loss: a common photometric yardstick, not any method's own objective
# (regularisers never look at the ground truth and have no evaluation counterpart).
EVAL_LOSS_KIND = CFG.get("eval_loss_kind", "l1")     # "l1" | "l1_dssim"
LAMBDA_DSSIM = float(CFG.get("lambda_dssim", 0.2))
LPIPS_NET = CFG.get("lpips_net", "vgg")
MAX_EVAL_VIEWS = int(CFG.get("max_eval_views", 0))   # 0 = all test views
EVAL_AT_FIRST_ITERATION = bool(CFG.get("eval_at_first_iteration", True))

# BATCH_SIZE: views per iteration (per-scene batch for fudan-zvg, 1 for the other two).
# ITERATION_OFFSET: iterations of an earlier stage whose counter restarted (coarse stage of Wu et al.).
BATCH_SIZE = int(CFG.get("batch_size", 1))
ITERATION_OFFSET = int(CFG.get("iteration_offset", 0))


# LPIPS: bundled lpipsPyTorch, falling back to the pip package; the model is built once.

_lpips_model = None
_lpips_backend = None


def _get_lpips():
    global _lpips_model, _lpips_backend
    if _lpips_model is not None:
        return _lpips_model, _lpips_backend
    try:
        from lpipsPyTorch.modules.lpips import LPIPS
        _lpips_model = LPIPS(net_type=LPIPS_NET).to("cuda").eval()
        _lpips_backend = "lpipsPyTorch"
    except Exception as exc:
        print("[benchmark] lpipsPyTorch unavailable (%r), falling back to the "
              "lpips pip package." % (exc,))
        import lpips as _lpips_pkg
        _lpips_model = _lpips_pkg.LPIPS(net=LPIPS_NET).to("cuda").eval()
        _lpips_backend = "lpips-pip"
    return _lpips_model, _lpips_backend


def _lpips_value(image, gt):
    model, backend = _get_lpips()
    x = image.unsqueeze(0)
    y = gt.unsqueeze(0)
    if backend == "lpips-pip":
        # the pip package expects inputs in [-1, 1]
        x = x * 2.0 - 1.0
        y = y * 2.0 - 1.0
    return model(x, y).mean().item()


# JSON summary, rewritten atomically at every evaluation

_SUMMARY = {
    "method": METHOD,
    "scene": SCENE,
    "mode": MODE,
    "config": CFG,
    "status": "running",
    "target_reached": False,
    "entries": [],
    "final": None,
}


def _flush_summary():
    directory = os.path.dirname(JSON_PATH)
    if directory:
        os.makedirs(directory, exist_ok=True)
    tmp_path = JSON_PATH + ".tmp"
    with open(tmp_path, "w") as handle:
        json.dump(_SUMMARY, handle, indent=2)
    os.replace(tmp_path, JSON_PATH)


@atexit.register
def _on_exit():
    if _SUMMARY["status"] == "running":
        _SUMMARY["status"] = "finished"
    if _SUMMARY["entries"]:
        _SUMMARY["final"] = _SUMMARY["entries"][-1]
    try:
        _flush_summary()
    except Exception as exc:
        print("[benchmark] could not write the JSON summary: %r" % (exc,))


class _Monitor(object):

    def __init__(self):
        self.t_start = None
        self.eval_overhead_s = 0.0
        self.last_eval_time = None
        self.last_eval_iteration = -1
        self.best_psnr = -1.0
        self.n_evals = 0
        self.consecutive_hits = 0

    def _target_met(self, entry):
        if entry["iteration"] < MIN_ITERATIONS_BEFORE_STOP:
            return False
        if TARGET_METRIC == "psnr":
            return entry["psnr"] >= TARGET_PSNR
        return entry["eval_loss"] <= TARGET_EVAL_LOSS

    def _must_evaluate(self, iteration, now):
        if iteration == self.last_eval_iteration:
            return False
        if EVAL_AT_FIRST_ITERATION and self.n_evals == 0:
            return True
        if MODE == "iterations":
            if iteration >= MAX_ITERATIONS:
                return True
            return (iteration % EVAL_EVERY_N_ITERS) == 0
        # MODE == "target_eval_loss": time-driven sampling
        if iteration >= SAFETY_MAX_ITERATIONS:
            return True
        return (now - self.last_eval_time) >= EVAL_EVERY_N_SECONDS

    @torch.no_grad()
    def _evaluate(self, a):
        psnr_sum, ssim_sum, lpips_sum, l1_sum, n_views = 0.0, 0.0, 0.0, 0.0, 0
        eval_h, eval_w = 0, 0
        for view in _iter_eval_views(a):
            image, gt = _render_pair(a, view)
            eval_h, eval_w = int(image.shape[-2]), int(image.shape[-1])
            image_b = image.unsqueeze(0)
            gt_b = gt.unsqueeze(0)
            psnr_sum += _repo_psnr(image_b, gt_b).mean().item()
            ssim_sum += _repo_ssim(image, gt).mean().item()
            l1_sum += _repo_l1(image, gt).mean().item()
            lpips_sum += _lpips_value(image, gt)
            n_views += 1
        n_views = max(n_views, 1)
        return {
            "psnr": psnr_sum / n_views,
            "ssim": ssim_sum / n_views,
            "lpips": lpips_sum / n_views,
            "eval_l1_loss": l1_sum / n_views,
            "num_eval_views": n_views,
            "eval_resolution": [eval_h, eval_w],
        }

    def step(self, a):
        now = time.time()
        if self.t_start is None:
            self.t_start = now
            self.last_eval_time = now

        iteration = int(a["iteration"])
        stage = a.get("stage", None)

        if not _stage_is_evaluable(stage):
            return
        if not self._must_evaluate(iteration, now):
            return

        eval_t0 = time.time()
        torch.cuda.empty_cache()
        metrics = self._evaluate(a)
        torch.cuda.synchronize()
        eval_dt = time.time() - eval_t0

        # Both loss variants are always recorded (SSIM is computed anyway)
        eval_photometric = ((1.0 - LAMBDA_DSSIM) * metrics["eval_l1_loss"]
                            + LAMBDA_DSSIM * (1.0 - metrics["ssim"]))
        eval_loss = (eval_photometric if EVAL_LOSS_KIND == "l1_dssim"
                     else metrics["eval_l1_loss"])

        wall = eval_t0 - self.t_start
        entry = {
            "iteration": iteration,
            "total_iterations": iteration + ITERATION_OFFSET,
            "images_seen": (iteration + ITERATION_OFFSET) * BATCH_SIZE,
            "stage": stage if stage is not None else "single_stage",
            "training_time_s": wall - self.eval_overhead_s,
            "wall_time_s": wall,
            "benchmark_overhead_s": self.eval_overhead_s,
            "psnr": metrics["psnr"],
            "ssim": metrics["ssim"],
            "lpips": metrics["lpips"],
            "eval_l1_loss": metrics["eval_l1_loss"],
            "eval_photometric_loss": eval_photometric,
            "eval_loss": eval_loss,
            "eval_loss_kind": EVAL_LOSS_KIND,
            "num_eval_views": metrics["num_eval_views"],
            "eval_resolution": metrics["eval_resolution"],
            "num_gaussians": int(a["scene"].gaussians.get_xyz.shape[0]),
            "train_batch_loss": float(a["loss"].item()) if hasattr(a.get("loss", None), "item") else None,
            "eval_duration_s": eval_dt,
        }

        self.last_eval_iteration = iteration
        self.n_evals += 1

        _SUMMARY["entries"].append(entry)
        _SUMMARY["final"] = entry
        _flush_summary()

        print("\n[benchmark] iter %7d | train %8.1fs | PSNR %6.3f | SSIM %.4f | "
              "LPIPS %.4f | eval_loss %.5f | #G %d | eval@%dx%d"
              % (iteration, entry["training_time_s"], entry["psnr"], entry["ssim"],
                 entry["lpips"], entry["eval_loss"], entry["num_gaussians"],
                 entry["eval_resolution"][1], entry["eval_resolution"][0]),
              flush=True)

        _maybe_track_best(a, entry, self)

        # Evaluation, JSON flush and best-checkpoint writes are benchmark overhead, not training time.
        self.eval_overhead_s += time.time() - eval_t0
        self.last_eval_time = time.time()

        if MODE == "target_eval_loss":
            if self._target_met(entry):
                self.consecutive_hits += 1
                if TARGET_METRIC == "psnr":
                    print("[benchmark] target met (%d/%d consecutive): PSNR %.4f >= %.4f"
                          % (self.consecutive_hits, TARGET_CONSECUTIVE_HITS,
                             entry["psnr"], TARGET_PSNR), flush=True)
                else:
                    print("[benchmark] target met (%d/%d consecutive): eval_loss "
                          "%.6f <= %.6f" % (self.consecutive_hits,
                                            TARGET_CONSECUTIVE_HITS,
                                            eval_loss, TARGET_EVAL_LOSS), flush=True)
            else:
                if self.consecutive_hits:
                    print("[benchmark] target lost again, consecutive counter reset.",
                          flush=True)
                self.consecutive_hits = 0

            if self.consecutive_hits >= TARGET_CONSECUTIVE_HITS:
                criterion = ("PSNR >= %.4f" % TARGET_PSNR if TARGET_METRIC == "psnr"
                             else "eval_loss <= %.6f" % TARGET_EVAL_LOSS)
                print("\n[benchmark] TARGET REACHED (%s held for %d consecutive "
                      "sampling points) at iteration %d. Saving the model and "
                      "stopping training." % (criterion, TARGET_CONSECUTIVE_HITS,
                                              iteration), flush=True)
                _SUMMARY["target_reached"] = True
                _SUMMARY["status"] = "stopped_on_target"
                _SUMMARY["target_metric"] = TARGET_METRIC
                _save_model(a)
                _flush_summary()
                sys.stdout.flush()
                sys.exit(0)
            if iteration >= SAFETY_MAX_ITERATIONS:
                print("\n[benchmark] Safety cap of %d iterations reached without "
                      "hitting the target eval loss. Saving and stopping."
                      % SAFETY_MAX_ITERATIONS, flush=True)
                _SUMMARY["status"] = "stopped_on_safety_cap"
                _save_model(a)
                _flush_summary()
                sys.stdout.flush()
                sys.exit(0)


_MONITOR = _Monitor()


# Hook factory, called from the block injected at the bottom of train.py

def make_hook(original_training_report):
    signature = inspect.signature(original_training_report)

    def hooked_training_report(*args, **kwargs):
        result = original_training_report(*args, **kwargs)
        try:
            bound = signature.bind(*args, **kwargs)
            bound.apply_defaults()
            _MONITOR.step(bound.arguments)
        except SystemExit:
            raise
        except Exception as exc:
            print("[benchmark] evaluation hook failed: %r" % (exc,), flush=True)
        return result

    print("[benchmark] monitor installed (method=%s, scene=%s, mode=%s)"
          % (METHOD, SCENE, MODE), flush=True)
    return hooked_training_report


# Repository-specific glue: fudan-zvg/4d-gaussian-splatting. Test cameras yield (gt, camera)
# pairs; chkpnt_best.pth is kept on the benchmark schedule so render.py always finds it.

def _stage_is_evaluable(stage):
    return True


def _iter_eval_views(a):
    cameras = a["scene"].getTestCameras()
    total = len(cameras)
    indices = list(range(total))
    if MAX_EVAL_VIEWS and MAX_EVAL_VIEWS < total:
        step = max(1, total // MAX_EVAL_VIEWS)
        indices = indices[::step][:MAX_EVAL_VIEWS]
    for index in indices:
        yield cameras[index]


def _render_pair(a, item):
    gt_image, viewpoint = item
    gt = torch.clamp(gt_image.cuda(), 0.0, 1.0)[0:3, :, :]
    viewpoint = viewpoint.cuda()
    render_pkg = a["renderFunc"](viewpoint, a["scene"].gaussians, *a["renderArgs"])
    image = torch.clamp(render_pkg["render"], 0.0, 1.0)
    return image, gt


def _save_checkpoint(a, filename):
    scene = a["scene"]
    iteration = int(a["iteration"])
    path = os.path.join(scene.model_path, filename)
    torch.save((scene.gaussians.capture(), iteration), path)
    return path


def _save_model(a):
    iteration = int(a["iteration"])
    a["scene"].save(iteration)
    path = _save_checkpoint(a, "chkpnt_best.pth")
    print("[benchmark] checkpoint saved at iteration %d (%s)"
          % (iteration, path), flush=True)


def _maybe_track_best(a, entry, monitor):
    # Mirrors the repository rule: keep the best test-PSNR checkpoint.
    if entry["psnr"] >= monitor.best_psnr:
        monitor.best_psnr = entry["psnr"]
        _SUMMARY["best_psnr"] = entry["psnr"]
        _SUMMARY["best_psnr_iteration"] = entry["iteration"]
        _save_checkpoint(a, "chkpnt_best.pth")
        print("[benchmark] new best checkpoint (chkpnt_best.pth) at iteration %d"
              % entry["iteration"], flush=True)
'''

with open(os.path.join(REPO_DIR, "benchmark_monitor.py"), "w") as handle:
    handle.write(MONITOR_SRC)

# Inject the hook into train.py (idempotent; a pristine copy is kept as train.py.orig)
TRAIN_PY = os.path.join(REPO_DIR, "train.py")
BACKUP_PY = TRAIN_PY + ".orig"
if not os.path.exists(BACKUP_PY):
    shutil.copyfile(TRAIN_PY, BACKUP_PY)
shutil.copyfile(BACKUP_PY, TRAIN_PY)

with open(TRAIN_PY, "r") as handle:
    source = handle.read()

INJECTED = (
    "# === BENCHMARK HOOK (injected by the notebook) ===\n"
    "import os as _bench_os\n"
    "if _bench_os.environ.get('BENCH_CONFIG'):\n"
    "    import benchmark_monitor as _bench\n"
    "    training_report = _bench.make_hook(training_report)\n"
    "# === END BENCHMARK HOOK ===\n\n"
)
ANCHOR = 'if __name__ == "__main__":'
assert source.count(ANCHOR) == 1, "Unexpected train.py layout."
with open(TRAIN_PY, "w") as handle:
    handle.write(source.replace(ANCHOR, INJECTED + ANCHOR, 1))

print("benchmark_monitor.py written and hook injected into train.py.")
print("Without the BENCH_CONFIG environment variable train.py behaves exactly "
      "as upstream (pristine copy kept as train.py.orig).")


# Peak VRAM: training runs in a subprocess, so nvidia-smi is polled for the whole device.
class VramSampler(threading.Thread):

    def __init__(self, period_s=2.0):
        threading.Thread.__init__(self)
        self.daemon = True
        self.period_s = period_s
        self.peak_mb = 0.0
        self._stop_event = threading.Event()

    def run(self):
        while not self._stop_event.is_set():
            try:
                out = subprocess.check_output(
                    ["nvidia-smi", "--query-gpu=memory.used",
                     "--format=csv,noheader,nounits"])
                self.peak_mb = max(self.peak_mb,
                                   float(out.decode().strip().splitlines()[0]))
            except Exception:
                pass
            self._stop_event.wait(self.period_s)

    def stop(self):
        self._stop_event.set()
        return self.peak_mb


def write_bench_config(p):
    """Serialise the benchmark settings read by benchmark_monitor.py."""
    target = TARGET_EVAL_LOSS_PER_SCENE.get(p["scene"])
    config = {
        "method": METHOD_NAME,
        "scene": p["scene"],
        "mode": p["mode"],
        "json_path": p["bench_json"],
        "eval_every_n_iters": int(EVAL_EVERY_N_ITERS),
        "eval_every_n_seconds": float(EVAL_EVERY_N_MINUTES) * 60.0,
        "max_iterations": int(MAX_ITERATIONS),
        "safety_max_iterations": int(SAFETY_MAX_ITERATIONS),
        "target_metric": TARGET_METRIC,
        "target_psnr": float(TARGET_PSNR),
        "target_eval_loss": float(target) if target is not None else 0.0,
        "min_iterations_before_stop": int(MIN_ITERATIONS_BEFORE_STOP),
        "target_consecutive_hits": int(TARGET_CONSECUTIVE_HITS),
        "eval_loss_kind": EVAL_LOSS_KIND,
        "lambda_dssim": float(LAMBDA_DSSIM),
        "lpips_net": LPIPS_NET,
        "max_eval_views": int(MAX_EVAL_VIEWS),
        "eval_at_first_iteration": bool(EVAL_AT_FIRST_ITERATION),
        "source_path": p["source_path"],
        "model_path": p["model_path"],
        "batch_size": int(BATCH_SIZE),
        "iteration_offset": int(globals().get("COARSE_ITERATIONS", 0)),
    }
    os.makedirs(p["benchmark_dir"], exist_ok=True)
    with open(p["bench_config"], "w") as handle:
        json.dump(config, handle, indent=2)
    return config


def record_run_cost(p, wall_s, peak_vram_mb):
    if not os.path.exists(p["bench_json"]):
        return
    with open(p["bench_json"], "r") as handle:
        summary = json.load(handle)
    summary["process_wall_time_s"] = wall_s
    summary["peak_vram_mb_nvidia_smi"] = peak_vram_mb
    with open(p["bench_json"], "w") as handle:
        json.dump(summary, handle, indent=2)


def train_scene(scene, mode=None):
    """Train one scene under one protocol. Returns a short status string.

    The loop needs this to be a function, so the training command is issued
    with subprocess rather than the `!` magic (which cannot appear inside a
    function body). Output is streamed line by line, as `!` would do.
    """
    mode = mode or TRAINING_MODE
    p = activate_scene(scene, mode)

    if mode == "target_eval_loss" and TARGET_METRIC == "eval_loss":
        if TARGET_EVAL_LOSS_PER_SCENE.get(scene) is None:
            print("[%s] no target calibrated for this scene -> skipped." % scene)
            return "skipped_no_target"

    if SKIP_TRAINING_IF_TRAINED and run_is_complete(p):
        print("[%s] already benchmarked -> skipped (%s)."
              % (scene, p["bench_json"]))
        return "skipped_already_trained"

    os.chdir(REPO_DIR)
    extra = prepare_scene(scene, mode)
    write_bench_config(p)
    budget = MAX_ITERATIONS if mode == "iterations" else SAFETY_MAX_ITERATIONS
    cmd = build_train_command(p, budget, extra)

    print("=" * 78)
    print("[%s | %s] %s" % (scene, mode, cmd))
    print("=" * 78, flush=True)

    env = dict(os.environ, BENCH_CONFIG=p["bench_config"])
    sampler = VramSampler()
    sampler.start()
    t0 = time.time()
    proc = subprocess.Popen(cmd, shell=True, cwd=REPO_DIR, env=env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
    proc.wait()
    wall = time.time() - t0
    peak = sampler.stop()
    record_run_cost(p, wall, peak)

    # Measure model storage now, before the local model is discarded.
    try:
        storage = model_storage_report(p["model_path"])
        with open(p["bench_json"], "r") as handle:
            summary = json.load(handle)
        summary["storage"] = storage
        with open(p["bench_json"], "w") as handle:
            json.dump(summary, handle, indent=2)
        print("[%s] model storage %.2f MB (full folder %.2f MB)"
              % (scene, storage["model_storage_mb"], storage["full_folder_mb"]))
    except Exception as exc:
        print("[%s] storage measurement failed: %r" % (scene, exc))

    if KEEP_MODEL_ON_DRIVE:
        print("[%s] model kept at %s" % (scene, p["model_path"]))
    else:
        shutil.rmtree(p["model_path"], ignore_errors=True)
        print("[%s] local model discarded; metrics kept on Drive." % scene)


    print("\n[%s] finished in %dm %ds (return code %d), peak VRAM %.0f MB"
          % (scene, int(wall // 60), int(wall % 60), proc.returncode, peak),
          flush=True)
    return "ok" if proc.returncode == 0 else "failed(%d)" % proc.returncode


print("Training driver ready.")

### 4.1 Single-scene training

Trains `SCENE` under the protocol selected by `TRAINING_MODE`. Inert unless `RUN_MODE == "single"`.

In [ ]:
# 4.1  Train a single scene (inert unless RUN_MODE == "single")
if RUN_MODE != "single":
    print("RUN_MODE is 'loop': use cell 4.2 instead.")
else:
    status = train_scene(SCENE)
    print("Status: %s" % status)

### 4.2 Training loop

Trains every scene in `SCENES_TO_RUN` sequentially, one run fully written to storage before the next starts. Inert unless `RUN_MODE == "loop"`. The loop is resumable: completed runs are skipped, so re-running the cell after a disconnection or a GPU-quota stop continues where it stopped. A scene that raises an exception is recorded and the loop moves on. Scenes run one at a time on purpose, since sharing the GPU would corrupt the time and VRAM measurements.

In [ ]:
# 4.2  Train every scene in SCENES_TO_RUN (re-run the cell to resume after a disconnection)
import time

if RUN_MODE != "loop":
    print("RUN_MODE is 'single': use cell 4.1 instead.")
else:
    loop_results = {}
    loop_t0 = time.time()
    for _i, _scene in enumerate(SCENES_TO_RUN, 1):
        print("\n\n### [%d/%d] %s ###" % (_i, len(SCENES_TO_RUN), _scene),
              flush=True)
        try:
            loop_results[_scene] = train_scene(_scene)
        except KeyboardInterrupt:
            loop_results[_scene] = "interrupted"
            break
        except Exception as _exc:
            loop_results[_scene] = "error: %r" % (_exc,)
            print("[%s] ERROR: %r -- continuing with the next scene."
                  % (_scene, _exc), flush=True)

    print("\n" + "=" * 60)
    print(" LOOP SUMMARY (%s, %.1f min)"
          % (TRAINING_MODE, (time.time() - loop_t0) / 60.0))
    print("=" * 60)
    for _scene in SCENES_TO_RUN:
        print("  %-16s %s" % (_scene, loop_results.get(_scene, "not reached")))

### 4.3 Protocol B target calibration

Run after the Protocol A loop. Prints this method's final eval L1 per scene and the candidate target with a 5% margin. The target used in `TARGET_EVAL_LOSS_PER_SCENE` is the **largest candidate across the three methods**, identical in all three notebooks; the margin keeps the slowest method from meeting the target only where its curve flattens.

Note: the targets used in this study apply the same rule to the **minimum** of each Protocol A test curve rather than to the final sample. The two agree within 2% for Deformable-3DGS and 4DGaussians; for fudan-zvg the final sample is degraded by post-peak overfitting (see `docs/PROTOCOL_B_CALIBRATION.md`).

In [ ]:
# 4.3  Protocol B calibration: prints this method's candidate targets only. Take the per-scene
# maximum over the three methods and paste it into all three notebooks.
import json
import os

print("%-16s %14s %14s" % ("scene", "final eval L1", "candidate x1.05"))
print("-" * 48)
candidates = {}
for _scene in D_NERF_SCENES:
    _p = paths_for(_scene, "iterations")
    if not os.path.exists(_p["bench_json"]):
        print("%-16s %14s %14s" % (_scene, "-", "-"))
        continue
    with open(_p["bench_json"], "r") as handle:
        _summary = json.load(handle)
    _final = _summary.get("final")
    if not _final:
        print("%-16s %14s %14s" % (_scene, "-", "-"))
        continue
    _l1 = _final["eval_l1_loss"]
    candidates[_scene] = round(_l1 * 1.05, 6)
    print("%-16s %14.6f %14.6f" % (_scene, _l1, candidates[_scene]))

print("\nThis method's candidates (compare with the other two, keep the max):")
print(json.dumps(candidates, indent=4))

### 4.4 Benchmark summary

Reads the JSON of the active scene. After a loop, call `activate_scene("<scene>")` to inspect another one.

In [ ]:
# 4.4  Benchmark summary for the active scene
import json
import os

import pandas as pd

train_duration = None
benchmark_summary = None
benchmark_table = None

if os.path.exists(BENCH_JSON_PATH):
    with open(BENCH_JSON_PATH, "r") as handle:
        benchmark_summary = json.load(handle)
    entries = benchmark_summary.get("entries", [])
else:
    entries = []

if entries:
    _all = pd.DataFrame(entries)
    columns = [c for c in ["iteration", "total_iterations", "images_seen",
                           "training_time_s", "psnr", "ssim", "lpips",
                           "eval_loss", "eval_l1_loss", "eval_photometric_loss",
                           "num_gaussians", "stage"] if c in _all.columns]
    benchmark_table = _all[columns]
    pd.set_option("display.float_format", lambda v: "%.5f" % v)

    print("Method   : %s" % benchmark_summary["method"])
    print("Scene    : %s" % benchmark_summary["scene"])
    print("Protocol : %s" % benchmark_summary["mode"])
    print("Status   : %s" % benchmark_summary["status"])

    final = benchmark_summary["final"]
    train_duration = final["training_time_s"]

    if benchmark_summary["mode"] == "target_eval_loss":
        _cfg = benchmark_summary.get("config", {})
        _target = _cfg.get("target_eval_loss")
        print("Target   : eval_loss <= %.6f, held for %d consecutive samples"
              % (_target, _cfg.get("target_consecutive_hits", 2)))
        print("Reached  : %s" % benchmark_summary.get("target_reached"))
        # Report the first crossing of the target, not the termination (which includes the hysteresis).
        _first = next((e for e in entries if e["eval_loss"] <= _target), None)
        if _first is not None:
            print("\n  First crossing  : iteration %d, %.1f s (%.2f min)"
                  % (_first["iteration"], _first["training_time_s"],
                     _first["training_time_s"] / 60.0))
            print("  Stop confirmed  : iteration %d, %.1f s (%.2f min)"
                  % (final["iteration"], train_duration, train_duration / 60.0))
            print("  Hysteresis delay: %.1f s (+%.0f%%)"
                  % (train_duration - _first["training_time_s"],
                     100.0 * (train_duration / max(_first["training_time_s"], 1e-9) - 1.0)))
            print("  --> report the FIRST CROSSING as the cost of reaching the target.")
    print()
    display(benchmark_table)

    print()
    print("=" * 64)
    print(" FINAL POINT OF THE RUN")
    print("=" * 64)
    print("  Iterations                : %d" % final["iteration"])
    print("  Training time (net)       : %.2f s (%.2f min)"
          % (train_duration, train_duration / 60.0))
    print("  Benchmark overhead        : %.2f s" % final["benchmark_overhead_s"])
    print("  PSNR                      : %.4f dB" % final["psnr"])
    print("  SSIM                      : %.4f" % final["ssim"])
    print("  LPIPS                     : %.4f" % final["lpips"])
    print("  Eval loss (%-8s)      : %.6f" % (final["eval_loss_kind"], final["eval_loss"]))
    print("  Number of Gaussians       : %d" % final["num_gaussians"])
    if "eval_resolution" in final:
        print("  Evaluation resolution     : %d x %d  <-- must match across methods"
              % (final["eval_resolution"][1], final["eval_resolution"][0]))
    if "peak_vram_mb_nvidia_smi" in benchmark_summary:
        print("  Peak VRAM (device)        : %.0f MB"
              % benchmark_summary["peak_vram_mb_nvidia_smi"])
    print("=" * 64)
    print("JSON: %s" % BENCH_JSON_PATH)
else:
    print("No benchmark data for the active scene (%s, %s)." % (SCENE, TRAINING_MODE))

if benchmark_table is not None and len(benchmark_table) > 1:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
    minutes = benchmark_table["training_time_s"] / 60.0
    axes[0].plot(minutes, benchmark_table["psnr"], marker="o", color="tab:blue")
    axes[0].set_xlabel("training time [min]"); axes[0].set_ylabel("PSNR [dB]")
    axes[0].set_title("PSNR (higher is better)")
    axes[1].plot(minutes, benchmark_table["ssim"], marker="o", color="tab:green")
    axes[1].set_xlabel("training time [min]"); axes[1].set_ylabel("SSIM")
    axes[1].set_title("SSIM (higher is better)")
    axes[2].plot(minutes, benchmark_table["lpips"], marker="s", color="tab:red")
    axes[2].set_xlabel("training time [min]"); axes[2].set_ylabel("LPIPS")
    axes[2].set_title("LPIPS (lower is better)")
    axes[3].plot(benchmark_table["iteration"], benchmark_table["eval_loss"],
                 marker="o", color="tab:orange")
    axes[3].set_xlabel("iteration"); axes[3].set_ylabel("evaluation loss")
    axes[3].set_title("Evaluation loss (lower is better)")
    for ax in axes:
        ax.grid(alpha=0.3)
    fig.suptitle("%s -- %s (%s)" % (METHOD_NAME, SCENE, TRAINING_MODE))
    plt.tight_layout()
    plt.show()

---

# Part 2: evaluation and visualisation

**Do not run this part while a training loop is in progress.** Run cells 0.1 and 0.2 first (and the environment cells after a runtime restart), then call `activate_scene("<scene>")` to select the trained model. With `KEEP_MODEL_ON_DRIVE = False` the model exists only in the runtime that trained it.

---

## 5. Rendering and final metrics

The first cell cross-checks the model-storage figure stored in the benchmark JSON by classifying the checkpoint tensors independently. Then a community `render.py` (absent from the original repository) is written and run on `chkpnt_best.pth`, and PSNR, SSIM and LPIPS (VGG) are computed on the rendered test frames.

In [ ]:
# Independent check of the model-storage figure stored in the benchmark JSON
import json
import os

import torch

CKPT = os.path.join(MODEL_OUTPUT_PATH, "chkpnt_best.pth")
assert os.path.exists(CKPT), (
    "No checkpoint in %s -- run 4.1 with KEEP_MODEL_ON_DRIVE = True first."
    % MODEL_OUTPUT_PATH)

# capture() order for gaussian_dim == 4, from scene/gaussian_model.py
NAMES = ["active_sh_degree", "_xyz", "_features_dc", "_features_rest",
         "_scaling", "_rotation", "_opacity", "max_radii2D",
         "xyz_gradient_accum", "t_gradient_accum", "denom", "optimizer",
         "spatial_lr_scale", "_t", "_scaling_t", "_rotation_r",
         "rot_4d", "env_map", "active_sh_degree_t"]
MODEL_NAMES = {"_xyz", "_features_dc", "_features_rest", "_scaling",
               "_rotation", "_opacity", "_t", "_scaling_t", "_rotation_r"}

blob = torch.load(CKPT, map_location="cpu", weights_only=False)
state = blob[0] if isinstance(blob, (tuple, list)) else blob

assert len(state) == len(NAMES), (
    "Checkpoint has %d elements, expected %d: capture() changed and the index "
    "list in model_storage_report is no longer valid." % (len(state), len(NAMES)))


def nbytes(x):
    if torch.is_tensor(x):
        return x.numel() * x.element_size()
    if isinstance(x, dict):
        return sum(nbytes(v) for v in x.values())
    if isinstance(x, (list, tuple)):
        return sum(nbytes(v) for v in x)
    return 0


n_gauss = state[1].shape[0]
model_b = other_b = 0
print("%-3s %-20s %-14s %12s %s" % ("#", "name", "shape", "bytes", "class"))
print("-" * 68)
for i, (name, el) in enumerate(zip(NAMES, state)):
    b = nbytes(el)
    is_model = name in MODEL_NAMES
    shape = str(tuple(el.shape)) if torch.is_tensor(el) else type(el).__name__
    print("%-3d %-20s %-14s %12d %s"
          % (i, name, shape, b, "MODEL" if is_model else ""))
    # every model tensor must have one row per Gaussian
    if is_model:
        assert torch.is_tensor(el) and el.shape[0] == n_gauss, (
            "%s does not look like a per-Gaussian tensor: the index mapping is "
            "wrong." % name)
        model_b += b
    else:
        other_b += b

independent_mb = model_b / 1048576.0

with open(BENCH_JSON_PATH, "r") as handle:
    summary = json.load(handle)
reported = summary.get("storage")

print("\n" + "=" * 68)
print(" Gaussians                 : %d" % n_gauss)
print(" File on disk              : %.4f MB" % (os.path.getsize(CKPT) / 1048576.0))
print(" Model tensors (this cell) : %.4f MB" % independent_mb)
print(" Optimizer + accumulators  : %.4f MB" % (other_b / 1048576.0))
print(" Bytes per Gaussian        : %.1f" % (model_b / n_gauss))

if not reported:
    print("\n NO 'storage' FIELD IN THE JSON -> model_storage_report() never ran "
          "or raised. Check the training log for 'storage measurement failed'.")
else:
    print("\n Reported in JSON          : %.4f MB" % reported["model_storage_mb"])
    delta = abs(reported["model_storage_mb"] - independent_mb)
    print(" Difference                : %.6f MB" % delta)
    print("=" * 68)
    print(" VALIDATED: the reported figure matches the independent computation."
          if delta < 0.01 else
          " MISMATCH: model_storage_report is NOT measuring what it should.")
print("=" * 68)

In [ ]:
# Write render.py (community patch, not part of the original repository)
render_py_content = '''#
# Copyright (C) 2023, Inria
# GRAPHDECO research group, https://team.inria.fr/graphdeco
# All rights reserved.
#
# This software is free for non-commercial, research and evaluation use
# under the terms of the LICENSE.md file.
#
# For inquiries contact  george.drettakis@inria.fr
#

import torch
from scene import Scene
import os
from tqdm import tqdm
from os import makedirs
from gaussian_renderer import render
import torchvision
from utils.general_utils import safe_state
from argparse import ArgumentParser
from arguments import ModelParams, PipelineParams, get_combined_args
from gaussian_renderer import GaussianModel

def render_set(model_path, name, iteration, views, gaussians, pipeline, background):
    render_path = os.path.join(model_path, name, "ours_{}".format(iteration), "renders")
    gts_path = os.path.join(model_path, name, "ours_{}".format(iteration), "gt")

    makedirs(render_path, exist_ok=True)
    makedirs(gts_path, exist_ok=True)

    for idx, view in enumerate(tqdm(views, desc="Rendering progress")):
        rendering = render(view[1].cuda(), gaussians, pipeline, background)["render"]
        gt = view[0][0:3, :, :]
        torchvision.utils.save_image(rendering, os.path.join(render_path, \'{0:05d}\'.format(idx) + ".png"))
        torchvision.utils.save_image(gt, os.path.join(gts_path, \'{0:05d}\'.format(idx) + ".png"))

def render_sets(dataset : ModelParams, iteration : int, pipeline : PipelineParams, skip_train : bool, skip_test : bool):
    with torch.no_grad():
        gaussians = GaussianModel(dataset.sh_degree, gaussian_dim=4, rot_4d=True)
        scene = Scene(dataset, gaussians, shuffle=False)

        bg_color = [1,1,1] if dataset.white_background else [0, 0, 0]
        background = torch.tensor(bg_color, dtype=torch.float32, device="cuda")

        if not skip_train:
             render_set(dataset.model_path, "train", scene.loaded_iter, scene.getTrainCameras(), gaussians, pipeline, background)

        if not skip_test:
             render_set(dataset.model_path, "test", scene.loaded_iter, scene.getTestCameras(), gaussians, pipeline, background)

if __name__ == "__main__":
    parser = ArgumentParser(description="Testing script parameters")
    model = ModelParams(parser, sentinel=True)
    pipeline = PipelineParams(parser)
    parser.add_argument("--iteration", default=-1, type=int)
    parser.add_argument("--skip_train", action="store_true")
    parser.add_argument("--skip_test", action="store_true")
    parser.add_argument("--quiet", action="store_true")
    args = get_combined_args(parser)
    print("Rendering " + args.model_path)

    safe_state(args.quiet)

    render_sets(model.extract(args), args.iteration, pipeline.extract(args), args.skip_train, args.skip_test)
'''

with open("render.py", "w") as f:
    f.write(render_py_content)
print("render.py written successfully.")

# Patch scene/__init__.py to load the model from the .pth checkpoint instead of the .ply
import os
scene_init_path = "scene/__init__.py"
if os.path.exists(scene_init_path):
    with open(scene_init_path, "r") as f:
        scene_content = f.read()

    old_str = "self.gaussians.create_from_pth(args.loaded_pth, self.cameras_extent)"
    new_str = "self.gaussians.restore(model_args=torch.load(args.loaded_pth, weights_only=False)[0], training_args=None)"

    if old_str in scene_content:
        scene_content = scene_content.replace(old_str, new_str)
        with open(scene_init_path, "w") as f:
            f.write(scene_content)
        print("Patch applied to scene/__init__.py successfully (weights_only=False included).")
    elif new_str in scene_content:
        print("scene/__init__.py already patched.")
    else:
        print("The line to replace in scene/__init__.py was not found.")

In [ ]:
!python render.py --model_path {MODEL_DIR} --loaded_pth={MODEL_DIR}/chkpnt_best.pth

In [ ]:
import os
import glob
import torch
from PIL import Image
import torchvision.transforms.functional as TF
from torchmetrics.image import StructuralSimilarityIndexMeasure
from tqdm import tqdm

!pip install -q lpips
import lpips

base_test_dirs = glob.glob(os.path.join(MODEL_DIR, "test", "ours_*"))
if not base_test_dirs:
    raise FileNotFoundError("Could not find the test results folder.")

test_dir = base_test_dirs[0]
renders_dir = os.path.join(test_dir, "renders")
gt_dir = os.path.join(test_dir, "gt")

render_files = sorted(glob.glob(os.path.join(renders_dir, "*.png")))
gt_files = sorted(glob.glob(os.path.join(gt_dir, "*.png")))

print(f"Found {len(render_files)} rendered frames and {len(gt_files)} ground-truth frames.")

device = "cuda" if torch.cuda.is_available() else "cpu"

lpips_fn = lpips.LPIPS(net='vgg').to(device)
ssim_fn = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)

psnr_list, ssim_list, lpips_list = [], [], []

print("Computing metrics...")
for r_path, g_path in zip(tqdm(render_files), gt_files):
    r_img = TF.to_tensor(Image.open(r_path).convert("RGB")).unsqueeze(0).to(device)
    g_img = TF.to_tensor(Image.open(g_path).convert("RGB")).unsqueeze(0).to(device)

    mse = torch.mean((r_img - g_img) ** 2)
    psnr_val = -10.0 * torch.log10(mse)
    psnr_list.append(psnr_val.item())

    ssim_val = ssim_fn(r_img, g_img)
    ssim_list.append(ssim_val.item())

    # LPIPS expects inputs in [-1, 1]
    lpips_val = lpips_fn(r_img * 2.0 - 1.0, g_img * 2.0 - 1.0)
    lpips_list.append(lpips_val.item())

mean_psnr = sum(psnr_list) / len(psnr_list)
mean_ssim = sum(ssim_list) / len(ssim_list)
mean_lpips = sum(lpips_list) / len(lpips_list)

model_storage_mb = os.path.getsize(os.path.join(MODEL_DIR, "chkpnt_best.pth")) / (1024 ** 2)

print("\n" + "=" * 45)
print(" FINAL RESULTS ON THE TEST SET")
print("=" * 45)
if train_duration is not None:
    print(f"  Training time                : {train_duration:.2f} s ({train_duration/60:.2f} min)")
else:
    print("  Training time                : n/a (training was skipped in this session)")
print(f"  PSNR  (↑ higher is better)   : {mean_psnr:.4f} dB")
print(f"  SSIM  (↑ higher is better)   : {mean_ssim:.4f}")
print(f"  LPIPS (↓ lower is better)    : {mean_lpips:.4f}")
print(f"  Model storage                : {model_storage_mb:.2f} MB")
print("=" * 45)

## 6. Dense temporal rendering and video export

The test split has few discrete timestamps, so a video made from it lasts under a second. `render_dense.py` instead evaluates the model at many intermediate times (150 by default) and the frames are assembled into an MP4 at 30 FPS. `render_trajectory.py` additionally orbits the camera 360° while time advances and draws each moving ball's trajectory (found by K-Means on the fastest-moving Gaussians) as a fading trail. Both scripts read the true time range from the model (or the config's `time_duration`) instead of the hard-coded range of the original PR.

### 6.1 Write `render_dense.py`

In [ ]:
%cd /content/4d-gaussian-splatting

render_dense_py = r'''
import os
import torch
import torchvision
from tqdm import tqdm
from argparse import ArgumentParser

import gaussian_renderer.diff_gaussian_rasterization as dgr
import diff_gaussian_rasterization._C as _C_module
from scene import Scene
from gaussian_renderer import render, GaussianModel
from utils.general_utils import safe_state
from arguments import ModelParams, PipelineParams, OptimizationParams, get_combined_args


# Ensures tensors reaching the CUDA rasterizer are contiguous, without altering their values
_orig_c_rasterize = _C_module.rasterize_gaussians

def _safe_c_rasterize(*args):
    clean_args = []
    for arg in args:
        if isinstance(arg, torch.Tensor):
            t = arg.cuda() if not arg.is_cuda else arg
            t = t.contiguous()
            clean_args.append(t)
        else:
            clean_args.append(arg)
    return _orig_c_rasterize(*clean_args)

_C_module.rasterize_gaussians = _safe_c_rasterize
if hasattr(dgr, "_C"):
    dgr._C.rasterize_gaussians = _safe_c_rasterize


def render_dense(args, num_frames, camera_index=0):

    dataset = args
    pipeline = args

    cfg_path = os.path.join(dataset.model_path, "cfg_args")
    if os.path.exists(cfg_path):
        with open(cfg_path, "r") as f:
            cfg_args = eval(f.read(), {"Namespace": __import__("argparse").Namespace})
        for key, value in vars(cfg_args).items():
            setattr(args, key, value)

    gaussian_dim = getattr(args, "gaussian_dim", 4)
    rot_4d = getattr(args, "rot_4d", True)
    force_sh_3d = getattr(args, "force_sh_3d", False)
    time_duration = getattr(args, "time_duration", [0.0, 1.0])

    gaussians = GaussianModel(
        args.sh_degree,
        gaussian_dim=gaussian_dim,
        time_duration=time_duration,
        rot_4d=rot_4d,
        force_sh_3d=force_sh_3d,
        sh_degree_t=2
    )

    checkpoint_path = getattr(args, "loaded_pth", None)
    if not checkpoint_path or not os.path.exists(checkpoint_path):
        checkpoint_path = os.path.join(args.model_path, "chkpnt_best.pth")

    print(f"\nLoading checkpoint:\n{checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location="cuda", weights_only=False)
    model_params = checkpoint[0] if isinstance(checkpoint, tuple) else checkpoint
    gaussians.restore(model_params, args)
    print(f"Checkpoint restored. Number of Gaussians: {gaussians.get_xyz.shape[0]}")

    # Load the scene without resetting the point cloud
    orig_create_from_pcd = getattr(gaussians, "create_from_pcd", None)
    orig_load_ply = getattr(gaussians, "load_ply", None)

    if orig_create_from_pcd is not None:
        gaussians.create_from_pcd = lambda *a, **kw: None
    if orig_load_ply is not None:
        gaussians.load_ply = lambda *a, **kw: None

    scene = Scene(args, gaussians, shuffle=False)

    if orig_create_from_pcd is not None:
        gaussians.create_from_pcd = orig_create_from_pcd
    if orig_load_ply is not None:
        gaussians.load_ply = orig_load_ply

    train_cams = scene.getTrainCameras()
    test_cams = scene.getTestCameras()
    cameras = test_cams if len(test_cams) > 0 else train_cams

    if len(cameras) == 0:
        raise RuntimeError("No camera found in the scene.")

    cam_item = cameras[camera_index]
    camera = cam_item[1] if isinstance(cam_item, tuple) else cam_item

    # Detect the true time range (t_min, t_max) instead of assuming a fixed scale
    t_min, t_max = 0.0, 1.0

    if hasattr(gaussians, "_xyz") and gaussians._xyz.shape[-1] == 4:
        t_vals = gaussians._xyz[:, 3].detach().cpu()
        t_min, t_max = float(t_vals.min()), float(t_vals.max())
        print(f"Time range detected from _xyz (4D): min={t_min:.2f}, max={t_max:.2f}")
    elif hasattr(gaussians, "_xyz_t"):
        t_vals = gaussians._xyz_t.detach().cpu()
        t_min, t_max = float(t_vals.min()), float(t_vals.max())
        print(f"Time range detected from _xyz_t: min={t_min:.2f}, max={t_max:.2f}")
    else:
        train_times = [float(getattr(c[1] if isinstance(c, tuple) else c, 'timestamp', 0.0)) for c in train_cams]
        if len(train_times) > 0 and max(train_times) > 0:
            t_min, t_max = min(train_times), max(train_times)
            print(f"Time range detected from cameras: min={t_min:.2f}, max={t_max:.2f}")

    # White background, matching the Blender/D-NeRF convention
    background = torch.tensor([1, 1, 1], dtype=torch.float32, device="cuda")

    render_path = os.path.join(args.model_path, "dense_renders")
    os.makedirs(render_path, exist_ok=True)

    print(f"\nGenerating {num_frames} frames over the time range [{t_min:.2f}, {t_max:.2f}]...")
    time_values = torch.linspace(t_min, t_max, num_frames)

    with torch.no_grad():
        for frame_idx, t in enumerate(tqdm(time_values, desc="Temporal rendering")):
            t_val = float(t)
            t_tensor = torch.tensor([t_val], device="cuda", dtype=torch.float32)

            camera.timestamp = t_val
            camera.time = t_val
            if hasattr(camera, "fid"):
                camera.fid = t_val
            setattr(camera, "timestamp_tensor", t_tensor)
            setattr(camera, "time_tensor", t_tensor)

            rendering = render(camera, gaussians, pipeline, background)["render"]
            rendering = torch.clamp(rendering, 0.0, 1.0)

            torchvision.utils.save_image(
                rendering,
                os.path.join(render_path, f"{frame_idx:05d}.png")
            )

    print(f"\nRendering complete. Frames saved to:\n{render_path}")


if __name__ == "__main__":
    parser = ArgumentParser(conflict_handler="resolve")

    model = ModelParams(parser, sentinel=True)
    pipeline = PipelineParams(parser)
    optimization = OptimizationParams(parser)

    parser.add_argument("--loaded_pth", type=str, default=None)
    parser.add_argument("--iteration", default=-1, type=int)
    parser.add_argument("--num_frames", default=150, type=int)
    parser.add_argument("--camera_index", default=0, type=int)
    parser.add_argument("--quiet", action="store_true")

    args = get_combined_args(parser)
    safe_state(args.quiet)

    render_dense(args, args.num_frames, args.camera_index)
'''

with open("render_dense.py", "w") as f:
    f.write(render_dense_py)

print("render_dense.py written, with automatic time-range detection and white background.")

### 6.2 Run the dense rendering pass (150 frames)

In [ ]:
import os
import shutil

%cd /content/4d-gaussian-splatting

os.environ["TORCH_FORCE_WEIGHTS_ONLY_LOAD"] = "0"

CHECKPOINT_PATH = os.path.join(MODEL_DIR, "chkpnt_best.pth")

NUM_FRAMES = 150
CAMERA_INDEX = 0

RENDER_DIR = os.path.join(MODEL_DIR, "dense_renders")
if os.path.exists(RENDER_DIR):
    shutil.rmtree(RENDER_DIR)

print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Frames:     {NUM_FRAMES}")
print(f"Camera:     {CAMERA_INDEX}")

!python render_dense.py \
    --model_path {MODEL_DIR} \
    --loaded_pth {CHECKPOINT_PATH} \
    --num_frames {NUM_FRAMES} \
    --camera_index {CAMERA_INDEX}

### 6.3 Assemble the MP4 from the dense frames

In [ ]:
import os
import glob
import imageio
from IPython.display import HTML, display
from base64 import b64encode

RENDER_DIR = os.path.join(MODEL_DIR, "dense_renders")
OUTPUT_VIDEO = os.path.join(MODEL_DIR, "dense_render_%s.mp4" % SCENE)
FPS = 30  # 150 frames at 30 fps = 5 seconds of video

frames_path = sorted(glob.glob(os.path.join(RENDER_DIR, "*.png")))
if len(frames_path) == 0:
    raise FileNotFoundError(f"No .png frames found in {RENDER_DIR}")

print(f"Found {len(frames_path)} frames in {RENDER_DIR}")

writer = imageio.get_writer(OUTPUT_VIDEO, fps=FPS, codec="libx264", quality=8)
for img_path in frames_path:
    writer.append_data(imageio.imread(img_path))
writer.close()

print(f"\nVideo saved to:\n{OUTPUT_VIDEO}")

mp4_bytes = open(OUTPUT_VIDEO, 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4_bytes).decode()

display(HTML(f'''
<div style="text-align: center; margin-top: 10px;">
  <video controls autoplay loop width="600" style="border-radius: 8px; box-shadow: 0 4px 12px rgba(0,0,0,0.3);">
    <source src="{data_url}" type="video/mp4">
    Your browser does not support the video tag.
  </video>
  <p style="margin-top: 8px; font-size: 14px; color: #666;">
    <code>{os.path.basename(OUTPUT_VIDEO)}</code> (150 frames, 30 FPS)
  </p>
</div>
'''))

### 6.4 Write `render_trajectory.py` (orbiting camera and ball trails)

In [ ]:
%cd /content/4d-gaussian-splatting

render_traj_py = r'''
import os
import cv2
import copy
import torch
import numpy as np
import torchvision
from tqdm import tqdm
from argparse import ArgumentParser
from sklearn.cluster import KMeans

import gaussian_renderer.diff_gaussian_rasterization as dgr
import diff_gaussian_rasterization._C as _C_module
from scene import Scene
from gaussian_renderer import render, GaussianModel
from utils.general_utils import safe_state
from arguments import ModelParams, PipelineParams, OptimizationParams, get_combined_args

_orig_c_rasterize = _C_module.rasterize_gaussians
def _safe_c_rasterize(*args):
    clean_args = []
    for arg in args:
        if isinstance(arg, torch.Tensor):
            clean_args.append(arg.cuda().contiguous() if not arg.is_cuda else arg.contiguous())
        else:
            clean_args.append(arg)
    return _orig_c_rasterize(*clean_args)

_C_module.rasterize_gaussians = _safe_c_rasterize
if hasattr(dgr, "_C"):
    dgr._C.rasterize_gaussians = _safe_c_rasterize


def project_points(pts_3d, full_proj_matrix, W, H):
    """Project 3D points [N, 3] into 2D pixel coordinates [N, 2]."""
    pts_hom = torch.cat([pts_3d, torch.ones((pts_3d.shape[0], 1), device=pts_3d.device)], dim=-1)
    pts_cam = pts_hom @ full_proj_matrix

    w = pts_cam[:, 3:4]
    w = torch.clamp(w, min=1e-5)

    pts_2d = pts_cam[:, :2] / w

    u = (pts_2d[:, 0] + 1.0) * 0.5 * W
    v = (1.0 - (pts_2d[:, 1] + 1.0) * 0.5) * H
    return torch.stack([u, v], dim=-1)


def get_orbit_camera(cam_ref, angle_rad):
    """Return a copy of the camera rotated by angle_rad around the Z axis."""
    cam = copy.copy(cam_ref)

    cos_a, sin_a = np.cos(angle_rad), np.sin(angle_rad)
    R_z = torch.tensor([
        [cos_a, -sin_a, 0.0, 0.0],
        [sin_a,  cos_a, 0.0, 0.0],
        [0.0,    0.0,   1.0, 0.0],
        [0.0,    0.0,   0.0, 1.0]
    ], dtype=torch.float32, device="cuda")

    ref_world_view = cam_ref.world_view_transform.cuda()
    ref_full_proj = getattr(cam_ref, "full_proj_transform", None)
    if ref_full_proj is None:
        ref_full_proj = getattr(cam_ref, "full_projection_transform").cuda()
    else:
        ref_full_proj = ref_full_proj.cuda()

    cam.world_view_transform = R_z @ ref_world_view
    cam.full_proj_transform = R_z @ ref_full_proj

    if hasattr(cam_ref, "camera_center"):
        c_orig = cam_ref.camera_center.cuda()
        c_hom = torch.tensor([c_orig[0], c_orig[1], c_orig[2], 1.0], device="cuda")
        R_z_inv = torch.tensor([
            [cos_a, sin_a, 0.0, 0.0],
            [-sin_a, cos_a, 0.0, 0.0],
            [0.0, 0.0, 1.0, 0.0],
            [0.0, 0.0, 0.0, 1.0]
        ], dtype=torch.float32, device="cuda")
        cam.camera_center = (R_z_inv @ c_hom)[:3]

    return cam


def render_with_trajectories(args, num_frames=150, camera_index=0, num_balls=5, enable_orbit=True):

    dataset = args
    pipeline = args

    cfg_path = os.path.join(dataset.model_path, "cfg_args")
    if os.path.exists(cfg_path):
        with open(cfg_path, "r") as f:
            cfg_args = eval(f.read(), {"Namespace": __import__("argparse").Namespace})
        for key, value in vars(cfg_args).items():
            setattr(args, key, value)

    gaussians = GaussianModel(
        args.sh_degree,
        gaussian_dim=getattr(args, "gaussian_dim", 4),
        time_duration=getattr(args, "time_duration", [0.0, 1.0]),
        rot_4d=getattr(args, "rot_4d", True),
        force_sh_3d=getattr(args, "force_sh_3d", False),
        sh_degree_t=2
    )

    checkpoint_path = getattr(args, "loaded_pth", None) or os.path.join(args.model_path, "chkpnt_best.pth")
    print(f"\nLoading full checkpoint: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location="cuda", weights_only=False)

    if isinstance(checkpoint, tuple):
        model_params = checkpoint[0]
        deform_params = checkpoint[1] if len(checkpoint) > 1 else None
    else:
        model_params = checkpoint
        deform_params = None

    gaussians.restore(model_params, args)
    print(f"Point cloud restored: {gaussians.get_xyz.shape[0]} Gaussians.")

    if deform_params is not None:
        deform_obj = getattr(gaussians, "deform", getattr(gaussians, "_deform", None))
        if deform_obj is not None:
            if hasattr(deform_obj, "restore"):
                deform_obj.restore(deform_params)
                print("4D deformation module restored successfully (via .restore()).")
            elif hasattr(deform_obj, "load_state_dict"):
                if isinstance(deform_params, tuple):
                    deform_params = deform_params[0]
                deform_obj.load_state_dict(deform_params)
                print("4D deformation module restored successfully (via .load_state_dict()).")

    gaussians.create_from_pcd = lambda *a, **kw: None
    gaussians.load_ply = lambda *a, **kw: None

    scene = Scene(args, gaussians, shuffle=False)
    cameras = scene.getTestCameras() if len(scene.getTestCameras()) > 0 else scene.getTrainCameras()
    ref_cam = cameras[camera_index][1] if isinstance(cameras[camera_index], tuple) else cameras[camera_index]

    t_min, t_max = 0.0, 1.0
    if hasattr(gaussians, "_xyz") and gaussians._xyz.shape[-1] == 4:
        t_vals = gaussians._xyz[:, 3].detach().cpu()
        t_min, t_max = float(t_vals.min()), float(t_vals.max())

    time_samples = torch.linspace(t_min, t_max, num_frames, device="cuda")

    # Cluster the fastest-moving Gaussians to obtain one continuous trajectory per ball
    print("\nDetecting and clustering moving-ball centers...")
    deform_func = getattr(gaussians, "deform", None)

    with torch.no_grad():
        if deform_func is not None:
            t_check = torch.linspace(t_min, t_max, 8, device="cuda")
            positions_over_time = []
            for t_c in t_check:
                t_t = torch.tensor([float(t_c)], device="cuda", dtype=torch.float32)
                d_c = deform_func(gaussians.get_xyz, t_t)[0]
                positions_over_time.append(gaussians.get_xyz + d_c)

            pos_stack = torch.stack(positions_over_time, dim=0) # [8, N, 3]
            motion_magnitude = torch.std(pos_stack, dim=0).sum(dim=-1)
        else:
            motion_magnitude = torch.norm(gaussians.get_xyz[:, :3], dim=-1)

        # Take the top 5% fastest-moving points for K-Means clustering
        top_k_pts = max(100, int(gaussians.get_xyz.shape[0] * 0.05))
        _, moving_indices = torch.topk(motion_magnitude, k=top_k_pts)

        moving_xyz_0 = gaussians.get_xyz[moving_indices, :3].detach().cpu().numpy()

        k_clusters = min(num_balls, len(moving_xyz_0))
        kmeans = KMeans(n_clusters=k_clusters, random_state=42, n_init=10).fit(moving_xyz_0)
        labels = kmeans.labels_

        cluster_masks = []
        for c in range(k_clusters):
            c_mask = (labels == c)
            cluster_masks.append(moving_indices[c_mask])

    # Sample each ball's center of mass over time: [num_frames, K_balls, 3]
    trajectories_3d = []
    with torch.no_grad():
        for t in time_samples:
            t_tensor = torch.tensor([float(t)], device="cuda", dtype=torch.float32)
            ball_centers_t = []

            for c_idx in range(len(cluster_masks)):
                indices_c = cluster_masks[c_idx]
                if deform_func is not None:
                    d_xyz_c = deform_func(gaussians.get_xyz[indices_c], t_tensor)[0]
                    pos_c = gaussians.get_xyz[indices_c, :3] + d_xyz_c
                else:
                    pos_c = gaussians.get_xyz[indices_c, :3]

                center_t = pos_c.mean(dim=0)
                ball_centers_t.append(center_t)

            trajectories_3d.append(torch.stack(ball_centers_t, dim=0))

    trajectories_3d = torch.stack(trajectories_3d, dim=0) # [num_frames, K_balls, 3]

    background = torch.tensor([1, 1, 1], dtype=torch.float32, device="cuda")
    render_path = os.path.join(args.model_path, "trajectory_renders")
    os.makedirs(render_path, exist_ok=True)

    W, H = int(ref_cam.image_width), int(ref_cam.image_height)

    hues = [20, 55, 95, 135, 165]

    print(f"Generating {num_frames} frames with orbit camera and ball trails...")
    with torch.no_grad():
        for f_idx, t in enumerate(tqdm(time_samples, desc="Trajectory rendering")):
            t_val = float(t)
            t_tensor = torch.tensor([t_val], device="cuda", dtype=torch.float32)

            angle_rad = -2.0 * np.pi * (f_idx / num_frames) if enable_orbit else 0.0
            cam_curr = get_orbit_camera(ref_cam, angle_rad)

            cam_curr.timestamp = t_val
            cam_curr.time = t_val
            setattr(cam_curr, "timestamp_tensor", t_tensor)
            setattr(cam_curr, "time_tensor", t_tensor)

            render_img = render(cam_curr, gaussians, pipeline, background)["render"]
            render_img = torch.clamp(render_img, 0.0, 1.0)

            img_np = (render_img.permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
            img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)

            full_proj_curr = getattr(cam_curr, "full_proj_transform").cuda()
            curr_traj_3d = trajectories_3d[:f_idx+1] # [f_idx+1, K_balls, 3]

            for k in range(curr_traj_3d.shape[1]):
                pts_k_3d = curr_traj_3d[:, k, :]
                pts_k_2d = project_points(pts_k_3d, full_proj_curr, W, H).cpu().numpy()

                num_pts = len(pts_k_2d)
                if num_pts > 1:
                    hue = hues[k % len(hues)]
                    for i in range(1, num_pts):
                        p1 = (int(pts_k_2d[i-1, 0]), int(pts_k_2d[i-1, 1]))
                        p2 = (int(pts_k_2d[i, 0]), int(pts_k_2d[i, 1]))

                        # Skip off-screen or sub-pixel segments
                        dist_px = np.hypot(p2[0] - p1[0], p2[1] - p1[1])
                        if dist_px < 1.5 or not (0 <= p2[0] < W and 0 <= p2[1] < H):
                            continue

                        alpha = float(i) / num_pts
                        thickness = max(2, int(2 + 3 * alpha))
                        val = int(140 + 115 * alpha)
                        color_bgr = tuple(int(c) for c in cv2.cvtColor(np.uint8([[[hue, 255, val]]]), cv2.COLOR_HSV2BGR)[0,0])

                        cv2.line(img_bgr, p1, p2, color_bgr, thickness, lineType=cv2.LINE_AA)

            cv2.imwrite(os.path.join(render_path, f"{f_idx:05d}.png"), img_bgr)

    print(f"\nRendering complete. Frames saved to:\n{render_path}")


if __name__ == "__main__":
    parser = ArgumentParser(conflict_handler="resolve")
    model = ModelParams(parser, sentinel=True)
    pipeline = PipelineParams(parser)
    optimization = OptimizationParams(parser)

    parser.add_argument("--loaded_pth", type=str, default=None)
    parser.add_argument("--num_frames", default=150, type=int)
    parser.add_argument("--camera_index", default=0, type=int)
    parser.add_argument("--quiet", action="store_true")

    args = get_combined_args(parser)
    safe_state(args.quiet)

    render_with_trajectories(args, args.num_frames, args.camera_index)
'''

with open("render_trajectory.py", "w") as f:
    f.write(render_traj_py)

print("render_trajectory.py written.")

### 6.5 Run the trajectory rendering and assemble the MP4

In [ ]:
import os
import shutil
import glob
import imageio
from IPython.display import HTML, display
from base64 import b64encode

%cd /content/4d-gaussian-splatting

CHECKPOINT_PATH = os.path.join(MODEL_DIR, "chkpnt_best.pth")
RENDER_DIR = os.path.join(MODEL_DIR, "trajectory_renders")
OUTPUT_VIDEO = os.path.join(MODEL_DIR, "trajectory_%s.mp4" % SCENE)

if os.path.exists(RENDER_DIR):
    shutil.rmtree(RENDER_DIR)

!python render_trajectory.py \
    --model_path {MODEL_DIR} \
    --loaded_pth {CHECKPOINT_PATH} \
    --num_frames 150 \
    --camera_index 0

frames_path = sorted(glob.glob(os.path.join(RENDER_DIR, "*.png")))

if len(frames_path) > 0:
    writer = imageio.get_writer(OUTPUT_VIDEO, fps=30, codec="libx264", quality=8)
    for img_p in frames_path:
        writer.append_data(imageio.imread(img_p))
    writer.close()

    print(f"\nVideo saved to:\n{OUTPUT_VIDEO}")

    mp4_bytes = open(OUTPUT_VIDEO, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4_bytes).decode()

    display(HTML(f'''
    <div style="text-align: center; margin-top: 10px;">
      <video controls autoplay loop width="600" style="border-radius: 8px; box-shadow: 0 4px 12px rgba(0,0,0,0.3);">
        <source src="{data_url}" type="video/mp4">
      </video>
      <p style="margin-top: 8px; font-size: 14px; color: #555;">
        <code>{os.path.basename(OUTPUT_VIDEO)}</code> — 4D ball trajectories, orbiting camera, 360°
      </p>
    </div>
    '''))

### 6.6 Sanity check: `render.py` and `render_dense.py` on the same camera and frame

Shows both renders side by side, to confirm the dense renderer is consistent before trusting the full video.

In [ ]:
import os
import glob
import matplotlib.pyplot as plt
from PIL import Image

CHECKPOINT_PATH = os.path.join(MODEL_DIR, "chkpnt_best.pth")

%cd /content/4d-gaussian-splatting
!python render.py --model_path {MODEL_DIR} --loaded_pth {CHECKPOINT_PATH}

orig_renders = sorted(glob.glob(os.path.join(MODEL_DIR, "test", "ours_*", "renders", "*.png")))
if len(orig_renders) == 0:
    orig_renders = sorted(glob.glob(os.path.join(MODEL_DIR, "train", "ours_*", "renders", "*.png")))
if len(orig_renders) == 0:
    raise FileNotFoundError("render.py produced no images — check the logs above.")
orig_frame_path = orig_renders[0]

!python render_dense.py \
    --model_path {MODEL_DIR} \
    --loaded_pth {CHECKPOINT_PATH} \
    --num_frames 1 \
    --camera_index 0

custom_renders = sorted(glob.glob(os.path.join(MODEL_DIR, "dense_renders", "*.png")))
if len(custom_renders) == 0:
    raise FileNotFoundError("render_dense.py produced no images — check the logs above.")
custom_frame_path = custom_renders[0]

img_orig = Image.open(orig_frame_path)
img_custom = Image.open(custom_frame_path)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(img_orig)
axes[0].set_title("render.py (original)")
axes[0].axis("off")

axes[1].imshow(img_custom)
axes[1].set_title("render_dense.py (custom)")
axes[1].axis("off")

plt.tight_layout()
plt.show()

print(f"\nSizes — original: {img_orig.size}, custom: {img_custom.size}")

## 7. Real-time streaming viewer

Same approach as the other two notebooks: rendering happens server-side with the training rasterizer, and JPEG frames are streamed to the browser over WebSocket. Model loading and time handling are the ones validated in `render_dense.py` (checkpoint restore, `t_min`/`t_max` read from the 4D Gaussians, camera timestamp set before each `render()` call); an orbit camera lets the browser move freely in space.

In [ ]:
!pip install -q websockets

In [ ]:
# Reinstall pointops2 as a non-editable package (same issue as simple-knn)
import subprocess, sys, os

check = subprocess.run([sys.executable, "-c", "import pointops2_cuda"], capture_output=True, text=True)
if check.returncode != 0:
    print("pointops2_cuda not importable — reinstalling as non-editable...")
    os.chdir("/content/4d-gaussian-splatting/pointops2")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "."], check=True)
    os.chdir("/content/4d-gaussian-splatting")
    print("pointops2 reinstalled.")
else:
    print("pointops2_cuda already importable.")

%cd /content/4d-gaussian-splatting
import os, torch
import gaussian_renderer.diff_gaussian_rasterization as dgr
import diff_gaussian_rasterization._C as _C_module
from scene import Scene
from gaussian_renderer import render, GaussianModel
from utils.general_utils import safe_state
from arguments import ModelParams, PipelineParams, OptimizationParams, get_combined_args
from argparse import ArgumentParser

# Make tensors reaching the CUDA rasterizer contiguous (values unchanged); idempotent across re-runs.
if not hasattr(_C_module, "_true_original_rasterize_gaussians"):
    _C_module._true_original_rasterize_gaussians = _C_module.rasterize_gaussians

def _safe_c_rasterize(*args):
    clean_args = []
    for arg in args:
        if isinstance(arg, torch.Tensor):
            t = arg.cuda() if not arg.is_cuda else arg
            clean_args.append(t.contiguous())
        else:
            clean_args.append(arg)
    return _C_module._true_original_rasterize_gaussians(*clean_args)

_C_module.rasterize_gaussians = _safe_c_rasterize
# Patch dgr._C separately only when it is a different module object
if hasattr(dgr, "_C") and dgr._C is not _C_module:
    dgr._C.rasterize_gaussians = _safe_c_rasterize

parser = ArgumentParser(conflict_handler="resolve")
model_p = ModelParams(parser, sentinel=True)
pipeline_p = PipelineParams(parser)
optimization_p = OptimizationParams(parser)
parser.add_argument("--loaded_pth", type=str, default=None)
parser.add_argument("--iteration", default=-1, type=int)
parser.add_argument("--quiet", action="store_true")

CHECKPOINT_PATH = os.path.join(MODEL_DIR, "chkpnt_best.pth")

# get_combined_args() reads sys.argv; it does not accept an argv list
sys.argv = ["notebook", "--model_path", MODEL_DIR, "--loaded_pth", CHECKPOINT_PATH]
args = get_combined_args(parser)
safe_state(args.quiet)

cfg_path = os.path.join(MODEL_DIR, "cfg_args")
with open(cfg_path, "r") as f:
    cfg_args = eval(f.read(), {"Namespace": __import__("argparse").Namespace})
for key, value in vars(cfg_args).items():
    setattr(args, key, value)

gaussian_dim = getattr(args, "gaussian_dim", 4)
rot_4d = getattr(args, "rot_4d", True)
force_sh_3d = getattr(args, "force_sh_3d", False)
time_duration = getattr(args, "time_duration", [0.0, 1.0])

gaussians = GaussianModel(
    args.sh_degree, gaussian_dim=gaussian_dim, time_duration=time_duration,
    rot_4d=rot_4d, force_sh_3d=force_sh_3d, sh_degree_t=2
)

checkpoint = torch.load(CHECKPOINT_PATH, map_location="cuda", weights_only=False)
model_params = checkpoint[0] if isinstance(checkpoint, tuple) else checkpoint
gaussians.restore(model_params, args)
print(f"Checkpoint restored. Number of Gaussians: {gaussians.get_xyz.shape[0]}")

orig_create_from_pcd = getattr(gaussians, "create_from_pcd", None)
orig_load_ply = getattr(gaussians, "load_ply", None)
if orig_create_from_pcd is not None:
    gaussians.create_from_pcd = lambda *a, **kw: None
if orig_load_ply is not None:
    gaussians.load_ply = lambda *a, **kw: None

scene = Scene(args, gaussians, shuffle=False)

if orig_create_from_pcd is not None:
    gaussians.create_from_pcd = orig_create_from_pcd
if orig_load_ply is not None:
    gaussians.load_ply = orig_load_ply

train_cams = scene.getTrainCameras()
test_cams = scene.getTestCameras()
cameras = test_cams if len(test_cams) > 0 else train_cams
cam_item = cameras[0]
ref_camera = cam_item[1] if isinstance(cam_item, tuple) else cam_item

t_min, t_max = 0.0, 1.0
if hasattr(gaussians, "_xyz") and gaussians._xyz.shape[-1] == 4:
    t_vals = gaussians._xyz[:, 3].detach().cpu()
    t_min, t_max = float(t_vals.min()), float(t_vals.max())
print(f"Time range: min={t_min:.2f}, max={t_max:.2f}")
print(f"Reference camera: {ref_camera.image_width}x{ref_camera.image_height}")

In [ ]:
import math
import numpy as np
import torch
from utils.graphics_utils import getWorld2View2, getProjectionMatrix

class MiniCam:
    def __init__(self, width, height, fovy, fovx, znear, zfar, world_view_transform, full_proj_transform):
        self.image_width = width
        self.image_height = height
        self.FoVy = fovy
        self.FoVx = fovx
        self.znear = znear
        self.zfar = zfar
        self.world_view_transform = world_view_transform
        self.full_proj_transform = full_proj_transform
        view_inv = torch.inverse(self.world_view_transform)
        self.camera_center = view_inv[3][:3]

class OrbitCamera:
    def __init__(self, ref_cam, radius=4.0):
        self.image_width = ref_cam.image_width
        self.image_height = ref_cam.image_height
        self.FoVx = ref_cam.FoVx
        self.FoVy = ref_cam.FoVy
        self.znear = ref_cam.znear
        self.zfar = ref_cam.zfar
        self.center = np.array([0.0, 0.0, 0.0])
        self.radius = radius
        self.azimuth = 0.0
        self.elevation = 0.2

    def orbit(self, d_az, d_el):
        self.azimuth += d_az
        self.elevation = float(np.clip(self.elevation + d_el, -1.5, 1.5))

    def zoom(self, factor):
        self.radius = float(np.clip(self.radius * factor, 0.5, 20.0))

    def _position(self):
        x = self.radius * math.cos(self.elevation) * math.sin(self.azimuth)
        y = self.radius * math.sin(self.elevation)
        z = self.radius * math.cos(self.elevation) * math.cos(self.azimuth)
        return self.center + np.array([x, y, z])

    def build_camera(self):
        eye = self._position()
        forward = (self.center - eye)
        forward /= (np.linalg.norm(forward) + 1e-8)
        world_up = np.array([0.0, 1.0, 0.0])
        right = np.cross(forward, world_up)
        right /= (np.linalg.norm(right) + 1e-8)
        cam_up = np.cross(right, forward)

        c2w = np.eye(4, dtype=np.float64)
        c2w[:3, 0] = right
        c2w[:3, 1] = cam_up
        c2w[:3, 2] = -forward
        c2w[:3, 3] = eye

        # Same Blender/D-NeRF convention used by dataset_readers.py
        matrix = np.linalg.inv(c2w)
        R = -np.transpose(matrix[:3, :3])
        R[:, 0] = -R[:, 0]
        T = -matrix[:3, 3]

        world_view_transform = torch.tensor(getWorld2View2(R.astype(np.float32), T.astype(np.float32))).transpose(0, 1).cuda()
        projection_matrix = getProjectionMatrix(
            znear=self.znear, zfar=self.zfar, fovX=self.FoVx, fovY=self.FoVy
        ).transpose(0, 1).cuda()
        full_proj_transform = world_view_transform.unsqueeze(0).bmm(
            projection_matrix.unsqueeze(0)
        ).squeeze(0)

        return MiniCam(
            width=self.image_width, height=self.image_height,
            fovy=self.FoVy, fovx=self.FoVx,
            znear=self.znear, zfar=self.zfar,
            world_view_transform=world_view_transform,
            full_proj_transform=full_proj_transform
        )

orbit_cam = OrbitCamera(ref_camera, radius=4.0)
display("OrbitCamera ready.")

In [ ]:
import io
from PIL import Image

background = torch.tensor([1, 1, 1], dtype=torch.float32, device="cuda")

@torch.no_grad()
def render_frame_jpeg(orbit_cam, t_normalized, quality=80):
    cam = orbit_cam.build_camera()
    t_val = t_min + t_normalized * (t_max - t_min)
    t_tensor = torch.tensor([t_val], device="cuda", dtype=torch.float32)
    cam.timestamp = t_val
    cam.time = t_val
    cam.fid = t_val
    cam.timestamp_tensor = t_tensor
    cam.time_tensor = t_tensor

    render_pkg = render(cam, gaussians, args, background)
    image = render_pkg["render"]
    image = torch.clamp(image, 0, 1)
    image = (image * 255).byte().permute(1, 2, 0).cpu().numpy()
    buf = io.BytesIO()
    Image.fromarray(image).save(buf, format="JPEG", quality=quality)
    return buf.getvalue()

_test = render_frame_jpeg(orbit_cam, 0.0)
display(f"Test frame: {len(_test)/1024:.1f} KB")

In [ ]:
import asyncio, json, threading
import websockets

render_lock = threading.Lock()
current_time = 0.0
TARGET_FPS = 15

async def handle_client(websocket):
    print("Client connected.")
    try:
        async def receive_controls():
            global current_time
            async for message in websocket:
                try:
                    msg = json.loads(message)
                except json.JSONDecodeError:
                    continue
                t = msg.get("type")
                if t == "orbit":
                    orbit_cam.orbit(msg.get("d_azimuth", 0.0), msg.get("d_elevation", 0.0))
                elif t == "zoom":
                    orbit_cam.zoom(msg.get("factor", 1.0))
                elif t == "time":
                    current_time = float(msg.get("value", 0.0))

        async def stream_frames():
            while True:
                with render_lock:
                    frame = render_frame_jpeg(orbit_cam, current_time)
                await websocket.send(frame)
                await asyncio.sleep(1.0 / TARGET_FPS)

        await asyncio.gather(receive_controls(), stream_frames())
    except websockets.exceptions.ConnectionClosed:
        print("Client disconnected.")

async def start_server():
    async with websockets.serve(handle_client, "localhost", 8765, max_size=None):
        print("WebSocket server listening on ws://localhost:8765")
        await asyncio.Future()

def run_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(start_server())

threading.Thread(target=run_server, daemon=True).start()
display("Server started in the background.")

In [ ]:
import subprocess
from IPython.display import display
from google.colab.output import eval_js

http_proc = subprocess.Popen(
    ["python", "-m", "http.server", "8000", "--directory", "/content/4d-gaussian-splatting"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
display("HTTP server started on port 8000.")

# Colab proxies over HTTPS, so the browser must use wss:// through the proxied hostname.
ws_proxy_url = eval_js("google.colab.kernel.proxyPort(8765)")
ws_url = ws_proxy_url.rstrip("/").replace("https://", "wss://")
display(f"WebSocket URL to embed in the client: {ws_url}")

In [ ]:
from IPython.display import display

viewer_html = f"""<!DOCTYPE html>
<html><head><meta charset="utf-8"><title>4D-GS (fudan-zvg) Live Viewer</title>
<style>
body{{margin:0;background:#111;color:#eee;font-family:sans-serif;}}
#canvas{{display:block;cursor:grab;}}
#hud{{position:fixed;top:10px;left:10px;padding:8px 12px;background:rgba(0,0,0,.5);border-radius:6px;}}
#playBtn{{background:#2a7;border:none;color:white;font-size:14px;padding:4px 10px;border-radius:4px;cursor:pointer;margin-bottom:6px;}}
#playBtn.stop{{background:#c33;}}
</style></head><body>
<canvas id="canvas"></canvas>
<div id="hud">
  <div>Drag = orbit • Scroll = zoom</div>
  <button id="playBtn">▶ Play</button>
  <div>t = <span id="tValue">0.00</span></div>
  <input type="range" id="timeSlider" min="0" max="1" step="0.01" value="0">
</div>
<script>
const canvas = document.getElementById('canvas');
const ctx = canvas.getContext('2d');
const img = new Image();
let dragging=false, lastX=0, lastY=0;
const ws = new WebSocket("{ws_url}");
ws.binaryType = "blob";
ws.onopen = () => console.log("WebSocket connected");
ws.onerror = (e) => console.error("WebSocket error", e);
ws.onmessage = (e) => {{
  const url = URL.createObjectURL(e.data);
  img.onload = () => {{ canvas.width=img.width; canvas.height=img.height; ctx.drawImage(img,0,0); URL.revokeObjectURL(url); }};
  img.src = url;
}};
canvas.addEventListener('mousedown', e => {{ dragging=true; lastX=e.clientX; lastY=e.clientY; canvas.style.cursor='grabbing'; }});
window.addEventListener('mouseup', () => {{ dragging=false; canvas.style.cursor='grab'; }});
window.addEventListener('mousemove', e => {{
  if(!dragging) return;
  const dx=(e.clientX-lastX)*0.005, dy=(e.clientY-lastY)*0.005;
  lastX=e.clientX; lastY=e.clientY;
  ws.send(JSON.stringify({{type:"orbit", d_azimuth:-dx, d_elevation:dy}}));
}});
canvas.addEventListener('wheel', e => {{
  e.preventDefault();
  ws.send(JSON.stringify({{type:"zoom", factor: e.deltaY>0?1.1:0.9}}));
}}, {{passive:false}});

const slider = document.getElementById('timeSlider');
const tValue = document.getElementById('tValue');
const playBtn = document.getElementById('playBtn');

function sendTime(t) {{
  tValue.textContent = t.toFixed(2);
  ws.send(JSON.stringify({{type:"time", value: t}}));
}}

slider.addEventListener('input', () => {{ sendTime(parseFloat(slider.value)); }});

let playing = false;
let playInterval = null;
const PLAY_STEP = 0.01;
const PLAY_INTERVAL_MS = 50;

playBtn.addEventListener('click', () => {{
  playing = !playing;
  if (playing) {{
    playBtn.textContent = '⏸ Stop';
    playBtn.classList.add('stop');
    playInterval = setInterval(() => {{
      let t = parseFloat(slider.value) + PLAY_STEP;
      if (t > 1.0) t = 0.0;
      slider.value = t;
      sendTime(t);
    }}, PLAY_INTERVAL_MS);
  }} else {{
    playBtn.textContent = '▶ Play';
    playBtn.classList.remove('stop');
    clearInterval(playInterval);
  }}
}});
</script></body></html>"""

with open("/content/4d-gaussian-splatting/viewer_client.html", "w") as f:
    f.write(viewer_html)
display("Client HTML written with the proxied WebSocket URL and Play/Stop control.")

In [ ]:
from IPython.display import display
from google.colab.output import eval_js

page_url = eval_js("google.colab.kernel.proxyPort(8000)")
display(f"{page_url}/viewer_client.html")